# Hunt-and-Kill y Búsqueda no Informada

**Curso:** Inteligencia Artificial · **Referencia:** AIMA, capítulo 3  
**Modalidad:** individual · **Duración sugerida:** 3 semanas  
**Producto:** un cuaderno Jupyter con tres implementaciones comparables

## Problema

A partir del grafo producido por el generador Hunt-and-Kill, se implementan y comparan algoritmos de búsqueda no informada para encontrar un camino entre dos celdas.

El taller se desarrolla en tres versiones:

1. **Desde cero:** con estructuras estándar, sin bibliotecas de búsqueda o grafos.
2. **SimpleAI:** modelando el laberinto como `SearchProblem`.
3. **AIMA-Python:** modelándolo como una subclase de `Problem`.

El cuaderno contiene la primera versión, Lee y adaptadores para las dos bibliotecas. El punto 8 verifica resultados, el punto 9 transforma el grafo para introducir ciclos y costos y el punto 10 define el protocolo experimental de 30 semillas por configuración. SimpleAI y AIMA-Python requieren instalar o agregar sus bibliotecas al kernel antes de ejecutar sus adaptadores.

## Resultados de aprendizaje

Al finalizar, el estudiante estará en capacidad de:

- formular un entorno como espacio de estados;
- separar el problema de la estrategia de búsqueda;
- implementar búsqueda en árbol y búsqueda en grafo;
- justificar completitud, optimalidad y complejidad;
- adaptar un mismo problema a dos bibliotecas de IA;
- construir experimentos reproducibles;
- interpretar el algoritmo de Lee como BFS por frentes de onda;
- defender decisiones de diseño y diagnosticar errores sin depender de una solución externa.

## Condiciones y restricciones

- No se permite `networkx` ni funciones externas que ya resuelvan caminos.
- En la versión desde cero no se permite SimpleAI, AIMA-Python ni otra biblioteca de búsqueda.
- Los estados deben ser inmutables y utilizables como claves de diccionario.
- Cada algoritmo debe devolver solución y métricas, no solamente imprimirlas.
- Todas las versiones deben trabajar con el **mismo grafo**, inicio, meta y costos.
- Los resultados deben ser reproducibles mediante semillas.
- No se acepta una imagen como evidencia: deben entregarse estructuras de datos verificables.
- El estudiante debe citar cualquier recurso consultado y conservar una bitácora breve de decisiones.

### Evidencia de autoría y comprensión

La evaluación incluye:

1. historial de al menos seis hitos o commits;
2. una traza manual sobre un laberinto pequeño;
3. defensa oral individual de 5–8 minutos;
4. modificación en vivo de inicio, meta, semilla o costo;
5. explicación de una decisión que produjo un error y cómo se corrigió.

Durante la defensa se podrá solicitar reconstruir una parte pequeña del algoritmo sin consultar el cuaderno.

## 1. Código suministrado: generador Hunt-and-Kill

Este es el único algoritmo completo entregado. El resultado es un diccionario de adyacencia no dirigido:

```text
grafo[celda] = conjunto de celdas conectadas por un corredor
```

Una celda se representa como `(fila, columna)`.

In [ ]:
import random


class LaberintoHuntKill:
    """Genera un laberinto perfecto como grafo no dirigido."""

    DIRECCIONES = ((-1, 0), (1, 0), (0, -1), (0, 1))

    def __init__(self, filas=25, columnas=25, semilla=2026):
        self.filas = filas
        self.columnas = columnas
        self.rng = random.Random(semilla)
        self.grafo = {
            (fila, columna): set()
            for fila in range(filas)
            for columna in range(columnas)
        }

    def vecinos_geometricos(self, celda):
        fila, columna = celda
        for df, dc in self.DIRECCIONES:
            vecino = (fila + df, columna + dc)
            if (0 <= vecino[0] < self.filas and
                    0 <= vecino[1] < self.columnas):
                yield vecino

    def conectar(self, origen, destino):
        self.grafo[origen].add(destino)
        self.grafo[destino].add(origen)

    def generar(self):
        no_visitadas = set(self.grafo)
        actual = self.rng.choice(tuple(no_visitadas))
        no_visitadas.remove(actual)

        while no_visitadas:
            # KILL: avanzar aleatoriamente hacia una celda no visitada.
            libres = [
                vecino
                for vecino in self.vecinos_geometricos(actual)
                if vecino in no_visitadas
            ]

            if libres:
                siguiente = self.rng.choice(libres)
                self.conectar(actual, siguiente)
                no_visitadas.remove(siguiente)
                actual = siguiente
                continue

            # HUNT: localizar una celda libre vecina del árbol construido.
            candidatas = []
            for celda in no_visitadas:
                visitados = [
                    vecino
                    for vecino in self.vecinos_geometricos(celda)
                    if vecino not in no_visitadas
                ]
                if visitados:
                    candidatas.append((celda, visitados))

            actual, visitados = self.rng.choice(candidatas)
            vecino = self.rng.choice(visitados)
            self.conectar(actual, vecino)
            no_visitadas.remove(actual)

        return self.grafo

### Actividad 1 — Auditoría del generador

Antes de resolver el laberinto:

1. Explique las fases Hunt y Kill señalando las instrucciones correspondientes.
2. Demuestre que el grafo generado es conexo.
3. Justifique por qué contiene exactamente $|V|-1$ aristas.
4. Explique por qué esas dos propiedades implican que es un árbol.
5. Diseñe pruebas para comprobar:
   - simetría de las adyacencias;
   - ausencia de conexiones diagonales;
   - validez de coordenadas;
   - conectividad;
   - ausencia de ciclos;
   - reproducibilidad con una semilla.

No basta con ejecutar el generador: las propiedades deben verificarse automáticamente.

**Respuesta y pruebas del estudiante:**

### 1. Fases Hunt y Kill

*(referidas a las líneas del código anterior)*

**KILL** — bloque comentado `# KILL: avanzar aleatoriamente...`

Mientras la celda `actual` tenga al menos un vecino geométrico que todavía esté en `no_visitadas`, el algoritmo se comporta como una **caminata aleatoria autoevitante**:

1. Elige uno de esos vecinos libres (`self.rng.choice(libres)`).
2. Lo conecta a `actual` (`self.conectar`).
3. Lo marca como visitado y lo convierte en la nueva celda `actual` (`continue` vuelve al inicio del `while`).

Esta fase construye "pasillos" en línea recta mientras hay territorio virgen alcanzable directamente.

**HUNT** — bloque comentado `# HUNT: localizar una celda libre...`

Se activa solo cuando KILL se atasca, es decir, cuando `actual` no tiene ningún vecino geométrico libre (`libres` queda vacío). Entonces:

1. Se recorren **todas** las celdas aún no visitadas (`for celda in no_visitadas`).
2. Para cada una, se listan sus vecinos que **ya** fueron visitados (`visitados`).
3. Cualquier celda con al menos un vecino visitado es candidata a "resucitar" la caminata.
4. Se elige una candidata al azar, se conecta con uno de sus vecinos ya visitados y se retoma como `actual`.

Esta fase es la que evita que el algoritmo quede atrapado: garantiza que siempre se pueda seguir creciendo el árbol mientras existan celdas sin visitar.



### 2. El grafo generado es conexo

Por construcción, toda arista que se agrega (`self.conectar`) une una celda que **ya** pertenece al árbol parcial (visitada, ergo alcanzable desde la raíz `actual` inicial) con una celda recién retirada de `no_visitadas`:

- En la fase **KILL**, `siguiente` se conecta a `actual` (visitada).
- En la fase **HUNT**, `actual` (la candidata, recién retirada de `no_visitadas`) se conecta con `vecino` (ya visitado).

Es decir, cada celda, en el momento en que deja de estar en `no_visitadas`, queda unida por una arista a una celda que ya era alcanzable desde la primera celda elegida.

> **Por inducción sobre el orden de remoción de `no_visitadas`:**
> la primera celda es trivialmente alcanzable (es la raíz); si las primeras *k* celdas removidas son alcanzables entre sí, la celda *k+1* se conecta a una de ellas, luego también es alcanzable.

Como el bucle `while no_visitadas` solo termina cuando el conjunto queda vacío, todas las celdas terminan siendo alcanzables desde la raíz: **el grafo es conexo**.



### 3. El grafo tiene exactamente |V| − 1 aristas

`self.conectar` es la **única** función que agrega aristas, y se invoca exactamente una vez por iteración exitosa del bucle exterior (una vez en KILL, una vez en HUNT, nunca ambas en la misma vuelta).

Cada llamada a `conectar` remueve exactamente una celda de `no_visitadas` (`no_visitadas.remove(...)`), y nunca se vuelve a insertar una celda ya removida (no hay `add` sobre `no_visitadas`).

Como `no_visitadas` arranca con |V| − 1 elementos (la raíz se remueve antes del bucle) y termina en 0, el bucle ejecuta exactamente |V| − 1 remociones, y por lo tanto exactamente **|V| − 1 llamadas a `conectar`**, es decir, |V| − 1 aristas no dirigidas.



### 4. Conexo + |V| − 1 aristas implica árbol

Es un resultado estándar de teoría de grafos (AIMA, cap. 3, y cualquier texto de grafos):

> Un grafo simple, no dirigido y conexo con *n* vértices es un árbol **si y solo si** tiene exactamente *n* − 1 aristas.

**Intuición:**

- Si tuviera **menos** de *n* − 1 aristas, no podría ser conexo (se necesitan al menos *n* − 1 aristas para conectar *n* vértices).
- Si teniendo *n* vértices conexos tuviera **más** de *n* − 1 aristas, existiría al menos una arista "sobrante" cuya eliminación no desconecta el grafo, lo cual implica que esa arista cerraba un ciclo.

Como el generador produce **exactamente** *n* − 1 aristas (punto 3) y el grafo es conexo (punto 2), no puede contener ciclos: es, por definición, un **árbol** (de ahí que se hable de *"laberinto perfecto"*: un único camino simple entre cualquier par de celdas).

In [ ]:
import unittest


class TestGeneradorHuntKill(unittest.TestCase):
    def setUp(self):
        self.filas, self.columnas = 8, 8
        self.grafo = LaberintoHuntKill(
            filas=self.filas, columnas=self.columnas, semilla=42
        ).generar()
        self.n_celdas = self.filas * self.columnas

    def test_simetria_de_adyacencias(self):
        for celda, vecinos in self.grafo.items():
            for vecino in vecinos:
                self.assertIn(celda, self.grafo[vecino])

    def test_sin_conexiones_diagonales(self):
        for (fila, columna), vecinos in self.grafo.items():
            for fila_vecino, columna_vecino in vecinos:
                self.assertEqual(
                    abs(fila - fila_vecino) + abs(columna - columna_vecino), 1
                )

    def test_coordenadas_validas(self):
        for fila, columna in self.grafo:
            self.assertTrue(0 <= fila < self.filas)
            self.assertTrue(0 <= columna < self.columnas)
        for vecinos in self.grafo.values():
            for fila, columna in vecinos:
                self.assertTrue(0 <= fila < self.filas)
                self.assertTrue(0 <= columna < self.columnas)

    def test_conectividad(self):
        raiz = next(iter(self.grafo))
        visitados = {raiz}
        pila = [raiz]
        while pila:
            actual = pila.pop()
            for vecino in self.grafo[actual]:
                if vecino not in visitados:
                    visitados.add(vecino)
                    pila.append(vecino)
        self.assertEqual(len(visitados), self.n_celdas)

    def test_arbol_tiene_n_menos_uno_aristas(self):
        aristas = sum(len(vecinos) for vecinos in self.grafo.values()) // 2
        self.assertEqual(aristas, self.n_celdas - 1)

    def test_ausencia_directa_de_ciclos(self):
        raiz = next(iter(self.grafo))
        visitados = set()

        def dfs(nodo, padre):
            visitados.add(nodo)
            for vecino in self.grafo[nodo]:
                if vecino == padre:
                    continue
                if vecino in visitados:
                    self.fail(f"Ciclo detectado en torno a {vecino}")
                dfs(vecino, nodo)

        dfs(raiz, None)
        self.assertEqual(len(visitados), self.n_celdas)

    def test_reproducibilidad_misma_semilla(self):
        grafo_a = LaberintoHuntKill(8, 8, 123).generar()
        grafo_b = LaberintoHuntKill(8, 8, 123).generar()
        self.assertEqual(grafo_a, grafo_b)


suite_generador = unittest.TestLoader().loadTestsFromTestCase(TestGeneradorHuntKill)
resultado_generador = unittest.TextTestRunner(verbosity=2).run(suite_generador)
assert resultado_generador.wasSuccessful(), "Fallaron las pruebas del generador"

## 2. Instancia individual

Cada estudiante debe construir una instancia reproducible:

- `semilla = últimos 5 dígitos del código estudiantil`;
- `filas = 20 + último dígito`;
- `columnas = 20 + penúltimo dígito`;
- inicio y meta deben estar en cuadrantes opuestos, pero no necesariamente en las esquinas.

Registre esos valores en una tabla. El docente podrá cambiar cualquiera de ellos durante la defensa.

| Parámetro | Valor |
|---|---|
| Semilla | 95022 |
| Filas | 22 |
| Columnas | 22 |
| Inicio | (21, 9) |
| Meta | (4, 12) |

In [ ]:
import random

semilla = 95022
filas = 20 + int(str(semilla)[-1])  
columnas = 20 + int(str(semilla)[-2])

generador = LaberintoHuntKill(filas=filas, columnas=columnas, semilla=semilla)
grafo = generador.generar()

# Elegir inicio y meta en cuadrantes opuestos (no necesariamente esquinas)
def cuadrante_de(celda, filas, columnas):
    fila, columna = celda
    mitad_fila, mitad_columna = filas / 2, columnas / 2
    if fila < mitad_fila and columna < mitad_columna:
        return 1
    if fila < mitad_fila and columna >= mitad_columna:
        return 2
    if fila >= mitad_fila and columna < mitad_columna:
        return 3
    return 4

cuadrante_opuesto = {1: 4, 2: 3, 3: 2, 4: 1}

# RNG independiente pero igual de reproducible (también derivado de la semilla).
rng_posiciones = random.Random(semilla + 1)
cuadrante_inicio = rng_posiciones.choice([1, 2, 3, 4])
cuadrante_meta = cuadrante_opuesto[cuadrante_inicio]

celdas_inicio = [c for c in grafo if cuadrante_de(c, filas, columnas) == cuadrante_inicio]
celdas_meta = [c for c in grafo if cuadrante_de(c, filas, columnas) == cuadrante_meta]

inicio = rng_posiciones.choice(celdas_inicio)
meta = rng_posiciones.choice(celdas_meta)

print("| Parámetro | Valor |")
print("|---|---|")
print(f"| Semilla   | {semilla} |")
print(f"| Filas     | {filas} |")
print(f"| Columnas  | {columnas} |")
print(f"| Inicio    | {inicio} |")
print(f"| Meta      | {meta} |")

## 3. Formulación formal del problema

Defina rigurosamente:

| Componente | Especificación por completar |
|---|---|
| Conjunto de estados       | Todas las celdas (fila, columna) con 0 <= fila < filas, 0 <= columna < columnas. Un estado es una tupla inmutable (fila, columna); coincide con los vértices del grafo generado por LaberintoHuntKill. |
| Estado inicial            | La celda `inicio` fijada en la instancia individual (sección 2).              |
| Acciones aplicables       | Dado un estado s, Acciones(s) = { mover_a(v) : v ∈ grafo[s] }. Es decir, moverse a cualquier celda directamente conectada a s por un corredor. El conjunto depende solo de las adyacencias del grafo, no de la posición geométrica de s en la cuadrícula. |
| Modelo de transición      | Resultado(s, mover_a(v)) = v, para v ∈ grafo[s]. Determinista: cada acción produce exactamente un sucesor. |
| Prueba de objetivo        | EsMeta(s)  ⇔  s = meta.                                                       |
| Costo de paso             | En la versión base (laberinto perfecto, sin ponderar): costo(s, a, s') = 1 para toda transición válida. En la extensión de la sección 9.B, costo(s, a, s') = peso(arista(s, s')) > 0, definido explícitamente por arista (ver esa sección). |
| Costo de una solución     | Suma de los costos de paso a lo largo del camino inicio → meta: g(meta) = Σ costo(sᵢ, aᵢ, sᵢ₊₁) para i = 0 .. n-1, donde s₀ = inicio y sₙ = meta. ||

Además:

1. diferencie **estado**, **nodo de búsqueda** y **celda dibujada**;
2. indique cuándo dos nodos distintos pueden contener el mismo estado;
3. explique por qué debe usarse búsqueda en grafo;
4. establezca invariantes que deben cumplirse durante una búsqueda.

## Respuesta a preguntas adicionales

### 1. Estado vs. nodo de búsqueda vs. celda dibujada

- **Estado:** la tupla `(fila, columna)` tal como aparece en el grafo. Es la unidad **matemática** del problema; es inmutable y hasheable.

- **Nodo de búsqueda:** estructura auxiliar utilizada por el algoritmo que **envuelve un estado** y añade metadatos de la búsqueda:
  - Padre.
  - Acción que lo generó.
  - Costo acumulado `g`.
  - Profundidad.

  Varios nodos de búsqueda distintos pueden envolver el mismo estado (ver pregunta 2).

- **Celda dibujada:** representación gráfica o visual del estado, por ejemplo, un píxel o un rectángulo coloreado en una figura de `matplotlib`. Se utiliza únicamente para mostrar el resultado a una persona.

  La celda dibujada **no participa en el algoritmo** y no debe utilizarse como fuente de verdad para ninguna prueba automática. De ahí la restricción de la sección **«Condiciones y restricciones»**: *«no se acepta una imagen como evidencia»*.

---

### 2. ¿Cuándo dos nodos distintos contienen el mismo estado?

Esto ocurre cuando el grafo **no es un árbol**, es decir, cuando existe más de un camino posible entre el estado inicial y un mismo estado.

Por ejemplo, esto puede ocurrir después de introducir ciclos en la sección **9.A**. En ese caso:

1. Dos ramas distintas de la búsqueda pueden llegar al mismo estado.
2. Cada rama puede haber seguido un camino diferente.
3. Por lo tanto, los caminos pueden tener costos acumulados `g` diferentes.
4. Se generan dos nodos de búsqueda distintos, con diferente padre y/o diferente `g`, que envuelven el mismo estado.

En el **laberinto perfecto original (árbol)** esto no puede ocurrir en una búsqueda en árbol sin repetición, porque solo existe un camino simple entre cualquier par de celdas.

Aun así, el mismo estado puede aparecer **transitoriamente en la frontera** si es generado como sucesor de dos nodos distintos antes de ser expandido.

---

### 3. ¿Por qué debe usarse búsqueda en grafo?

La **búsqueda en árbol**, al no tener control de repetidos, vuelve a expandir estados ya visitados cada vez que estos reaparecen como sucesores.

Esto presenta dos problemas:

- **En el laberinto base:** es innecesario, porque el grafo es un árbol y cualquier repetición simplemente representaría volver por donde se vino.
- **Cuando existen ciclos (sección 9.A):** el problema puede volverse explosivo, ya que el algoritmo podría recorrer el mismo ciclo indefinidamente y no terminar en un tiempo razonable.

La **búsqueda en grafo**, mediante un conjunto de estados `alcanzados` o `visitados`:

- Evita procesar repetidamente los mismos estados.
- Evita trabajo redundante.
- Garantiza la terminación en los casos correspondientes, evitando recorridos infinitos por ciclos.

La contrapartida es un **mayor consumo de memoria**, ya que es necesario almacenar los estados que ya han sido vistos.

---

### 4. Invariantes que deben cumplirse durante una búsqueda

Durante la ejecución del algoritmo deben mantenerse, como mínimo, los siguientes invariantes:

- **Alcanzabilidad:** todo estado que aparece en la frontera debe ser alcanzable desde el estado inicial mediante una secuencia válida de acciones.

- **Consistencia del costo:** `g(nodo)` debe ser exactamente la suma de los costos de los pasos a lo largo del camino registrado mediante los punteros a los padres. No debe ser un valor inventado o estar desincronizado del camino que puede reconstruirse.

- **Estados resueltos:** un estado no debe estar simultáneamente marcado como **definitivamente resuelto** (por ejemplo, expandido con costo óptimo en UCS) y seguir generando reemplazos con un costo peor.

- **Contador de expansiones:** `expandidos` solo debe incrementarse cuando un nodo se retira de la frontera **y se generan sus sucesores**, de acuerdo con el contrato del punto 4. No debe incrementarse dos veces para el mismo nodo.

- **Reconstrucción del camino:** cuando `encontrado = True`, el camino reconstruido debe comenzar en `inicio` y terminar en `meta`. Cuando `encontrado = False`, el camino debe ser vacío.


## 4. Contrato común de las tres versiones

Todos los algoritmos deben entregar un registro equivalente a:

```text
ResultadoBusqueda
    encontrado: booleano
    camino: secuencia de estados
    acciones: secuencia de acciones
    costo: número
    profundidad: entero
    expandidos: entero
    generados: entero
    repetidos_descartados: entero
    frontera_maxima: entero
    tiempo_ms: número
```

### Convenciones obligatorias

- Un estado se cuenta como **expandido** al retirarlo de la frontera y generar sus sucesores.
- Un nodo meta retirado de la frontera no se cuenta como expandido si no se generan sucesores.
- **Generado** significa nodo sucesor construido, aunque luego sea descartado.
- La política para estados repetidos debe documentarse.
- El camino debe incluir inicio y meta.
- Si no hay solución, `camino` debe ser vacío y `encontrado` falso.

Estas convenciones son necesarias para comparar implementaciones de manera justa.

## 5. Pseudocódigo base: búsqueda en grafo

El siguiente esquema no especifica la estructura de la frontera ni resuelve el manejo de costos. Debe especializarlo y justificar cada decisión.

```text
BUSQUEDA-GRAFO(problema, frontera):
    nodo_inicial ← crear_nodo(problema.estado_inicial)
    insertar(frente, nodo_inicial)
    alcanzados ← estructura apropiada

    mientras frontera no esté vacía:
        nodo ← extraer_segun_politica(frontera)

        si ES_META(nodo.estado):
            devolver RECONSTRUIR(nodo)

        si nodo debe expandirse:
            para cada acción aplicable:
                hijo ← construir sucesor
                decidir si insertar, reemplazar o descartar hijo

        actualizar métricas

    devolver FRACASO
```

### Preguntas de diseño

1. ¿Cuándo debe marcarse un estado: al generarlo o al expandirlo?
2. ¿La respuesta cambia entre BFS, DFS y UCS?
3. ¿Qué información mínima debe almacenar cada nodo?
4. ¿Cómo se reconstruye el camino sin copiar listas completas en cada nodo?
5. ¿Cómo se resuelve un camino más barato hacia un estado ya descubierto?

### 1. ¿Cuándo debe marcarse un estado, al generarlo o al expandirlo?

No hay una única respuesta correcta para todos los algoritmos:

- **Marcar al GENERAR:** evita que el mismo estado entre múltiples veces a la frontera. Esto produce una frontera más pequeña y requiere menos memoria.
- **Marcar al EXPANDIR:** es más simple de implementar, pero permite que copias duplicadas del mismo estado convivan temporalmente en la frontera antes de ser descartadas.

### 2. ¿La respuesta cambia entre BFS, DFS y UCS?

Sí:

- **BFS:** se marca al **GENERAR**. Como BFS expande por niveles y la primera vez que se alcanza un estado es siempre por el camino más corto (costo unitario), no tiene sentido permitir que ese estado vuelva a entrar a la cola. Marcarlo al generarlo evita crecimiento innecesario de la frontera sin sacrificar optimalidad.

- **DFS:** en esta implementación se marca al **EXPANDIR** (`alcanzados` se llena cuando el nodo se retira y se generan sucesores), porque a DFS no le preocupa el orden de llegada sino explorar en profundidad. Marcar al expandir es la variante más simple y es la que corresponde a "búsqueda en grafo" clásica de AIMA.

- **UCS:** **NO** basta con una marca binaria "visto/no visto". Un estado puede reaparecer con un costo `g` estrictamente menor al mejor conocido y, en ese caso, **SÍ debe reinsertarse con el costo actualizado**.

  Aquí se usa una marca condicional: se mantiene `mejor_costo[estado]` y solo se descarta un sucesor si su nuevo costo **NO mejora** `mejor_costo[estado]`.

### 3. ¿Qué información mínima debe almacenar cada nodo?

Cada nodo debe almacenar:

- **Estado**
- **Referencia al nodo padre** (no la lista completa de ancestros)
- **Acción que lo generó**
- **Costo acumulado `g`**
- **Profundidad**, opcionalmente

La profundidad se puede derivar de `padre.profundidad + 1`, pero almacenarla como atributo permite evitar recalcularla.

Ver la clase `Nodo` abajo.

### 4. ¿Cómo se reconstruye el camino sin copiar listas completas en cada nodo?

Cada nodo solo guarda un puntero al **nodo padre** y la **acción que lo generó**.

Para reconstruir el camino:

1. Se comienza desde el nodo objetivo.
2. Se sigue la referencia `padre` hacia atrás hasta llegar al nodo inicial.
3. Se almacenan las acciones encontradas.
4. Finalmente, se invierte la lista de acciones para obtener el camino desde el estado inicial hasta el objetivo.

De esta forma, no es necesario copiar la lista completa de ancestros en cada nodo.


# Versión 1 — Implementación desde cero

No puede utilizar bibliotecas de búsqueda o grafos. Se permiten únicamente estructuras estándar como listas, diccionarios, conjuntos, `deque` y colas de prioridad.

Debe implementar:

1. DFS iterativa;
2. BFS;
3. búsqueda de costo uniforme;
4. búsqueda limitada en profundidad;
5. profundización iterativa;
6. búsqueda bidireccional;
7. Lee o expansión por frente de onda.

La versión debe separar, como mínimo:

- representación del problema;
- representación de nodos;
- política de frontera;
- control de repetidos;
- reconstrucción del camino;
- recolección de métricas.

In [ ]:
from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple
import time
import itertools
import heapq

Celda = Tuple[int, int]
Grafo = Dict[Celda, set]


@dataclass
class ResultadoBusqueda:
    """Resultado comun: camino, costo y metricas de la busqueda."""
    encontrado: bool = False
    camino: List[Celda] = None
    acciones: List[Celda] = None
    costo: float = 0
    profundidad: int = 0
    expandidos: int = 0
    generados: int = 0
    repetidos_descartados: int = 0
    frontera_maxima: int = 0
    tiempo_ms: float = 0.0

    def __post_init__(self):
        if self.camino is None:
            self.camino = []
        if self.acciones is None:
            self.acciones = []


class ProblemaLaberinto:
    """Adaptador ligero de grafo de laberinto para la búsqueda desde cero."""

    def __init__(self, grafo: Grafo, inicio: Celda, meta: Celda,
                 costos: Optional[Dict[frozenset, float]] = None):
        self.grafo = grafo
        self.inicio = inicio
        self.meta = meta
        self.costos = costos

    def acciones(self, estado: Celda) -> List[Celda]:
        return sorted(self.grafo[estado])

    def resultado(self, estado: Celda, accion: Celda) -> Celda:
        if accion not in self.grafo[estado]:
            raise ValueError(f"Accion ilegal desde {estado}: {accion}")
        return accion

    def es_meta(self, estado: Celda) -> bool:
        return estado == self.meta

    def costo_paso(self, estado: Celda, accion: Celda, estado2: Celda) -> float:
        if estado2 not in self.grafo[estado] or accion != estado2:
            raise ValueError(f"Transicion ilegal: {estado} -> {estado2}")
        if self.costos is None:
            return 1.0
        return self.costos[frozenset((estado, estado2))]


@dataclass
class Nodo:
    estado: Celda
    padre: Optional["Nodo"]
    accion: Optional[Celda]
    g: float
    profundidad: int


def reconstruir_camino(nodo: Nodo) -> Tuple[List[Celda], List[Celda]]:
    camino: List[Celda] = []
    acciones: List[Celda] = []
    actual: Optional[Nodo] = nodo
    while actual is not None:
        camino.append(actual.estado)
        if actual.accion is not None:
            acciones.append(actual.accion)
        actual = actual.padre
    camino.reverse()
    acciones.reverse()
    return camino, acciones


class FronteraLIFO:
    """Pila LIFO para DFS."""
    def __init__(self):
        self._datos: List[Nodo] = []

    def insertar(self, nodo: Nodo) -> None:
        self._datos.append(nodo)

    def extraer(self) -> Nodo:
        return self._datos.pop()

    def vacia(self) -> bool:
        return not self._datos

    def __len__(self) -> int:
        return len(self._datos)


class FronteraFIFO:
    """Cola FIFO para BFS."""
    def __init__(self):
        self._datos: deque = deque()

    def insertar(self, nodo: Nodo) -> None:
        self._datos.append(nodo)

    def extraer(self) -> Nodo:
        return self._datos.popleft()

    def vacia(self) -> bool:
        return not self._datos

    def __len__(self) -> int:
        return len(self._datos)

## 6. Pseudocódigo que debe traducirse

### DFS y BFS

```text
inicializar frontera con el nodo inicial
inicializar alcanzados

mientras haya nodos:
    extraer según disciplina LIFO o FIFO
    comprobar objetivo
    expandir
    insertar estados nuevos
```

La diferencia no debe quedar dispersa por todo el programa. Diseñe una abstracción que permita cambiar la política de frontera.

### Costo uniforme

```text
frontera ← cola de prioridad ordenada por g(n)
mejor_costo[inicial] ← 0

mientras frontera no esté vacía:
    extraer nodo con menor g
    ignorar entradas obsoletas
    comprobar objetivo
    para cada sucesor:
        nuevo_costo ← g(nodo) + costo_de_paso
        si mejora el costo conocido:
            actualizar costo, padre y frontera
```

### Profundización iterativa

```text
para límite = 0, 1, 2, ...:
    resultado ← BUSQUEDA-LIMITADA(problema, límite)
    si resultado es solución: devolverlo
    si resultado es fracaso definitivo: terminar
```

Debe distinguir **corte** de **fracaso**.

### Bidireccional

```text
crear una frontera desde inicio y otra desde meta
expandir de forma equilibrada
detectar intersección entre regiones alcanzadas
unir los dos caminos respetando su orientación
```

No se acepta ejecutar dos búsquedas completas y comparar al final.

In [ ]:
# Busqueda DFS y BFS

def _busqueda_dfs_o_bfs(problema: ProblemaLaberinto, tipo: str) -> ResultadoBusqueda:
    """Comparte el control de busqueda y cambia solo la politica de frontera."""
    if tipo not in ("DFS", "BFS"):
        raise ValueError("tipo debe ser DFS o BFS")
    inicio_t = time.perf_counter()
    frontera = FronteraLIFO() if tipo == "DFS" else FronteraFIFO()
    nodo_inicial = Nodo(problema.inicio, None, None, 0, 0)
    frontera.insertar(nodo_inicial)
    alcanzados = {problema.inicio}
    expandidos_marca = set()
    expandidos = generados = repetidos_descartados = 0
    frontera_maxima = len(frontera)

    while not frontera.vacia():
        nodo = frontera.extraer()
        if tipo == "DFS" and nodo.estado in expandidos_marca:
            repetidos_descartados += 1
            continue
        if problema.es_meta(nodo.estado):
            camino, acciones = reconstruir_camino(nodo)
            return ResultadoBusqueda(
                encontrado=True, camino=camino, acciones=acciones,
                costo=nodo.g, profundidad=nodo.profundidad,
                expandidos=expandidos, generados=generados,
                repetidos_descartados=repetidos_descartados,
                frontera_maxima=frontera_maxima,
                tiempo_ms=(time.perf_counter() - inicio_t) * 1000,
            )
        if tipo == "DFS":
            expandidos_marca.add(nodo.estado)
        expandidos += 1
        for accion in problema.acciones(nodo.estado):
            sucesor = problema.resultado(nodo.estado, accion)
            generados += 1
            if tipo == "BFS":
                if sucesor in alcanzados:
                    repetidos_descartados += 1
                    continue
                alcanzados.add(sucesor)
            elif sucesor in expandidos_marca:
                repetidos_descartados += 1
                continue
            hijo = Nodo(
                estado=sucesor, padre=nodo, accion=accion,
                g=nodo.g + problema.costo_paso(nodo.estado, accion, sucesor),
                profundidad=nodo.profundidad + 1,
            )
            frontera.insertar(hijo)
        frontera_maxima = max(frontera_maxima, len(frontera))

    return ResultadoBusqueda(
        encontrado=False, expandidos=expandidos, generados=generados,
        repetidos_descartados=repetidos_descartados,
        frontera_maxima=frontera_maxima,
        tiempo_ms=(time.perf_counter() - inicio_t) * 1000,
    )


def busqueda_dfs(problema: ProblemaLaberinto) -> ResultadoBusqueda:
    return _busqueda_dfs_o_bfs(problema, "DFS")


def busqueda_bfs(problema: ProblemaLaberinto) -> ResultadoBusqueda:
    return _busqueda_dfs_o_bfs(problema, "BFS")

In [ ]:
# Busqueda de costo uniforme

def busqueda_costo_uniforme(problema: ProblemaLaberinto) -> ResultadoBusqueda:
    """UCS con heap, mejor costo por estado y descarte de entradas obsoletas."""
    inicio_t = time.perf_counter()
    contador = itertools.count()
    nodo_inicial = Nodo(problema.inicio, None, None, 0, 0)
    frontera: List[Tuple[float, int, Nodo]] = [(0, next(contador), nodo_inicial)]
    mejor_costo: Dict[Celda, float] = {problema.inicio: 0}
    expandidos = generados = repetidos_descartados = 0
    frontera_maxima = len(frontera)

    while frontera:
        frontera_maxima = max(frontera_maxima, len(frontera))
        costo_extraido, _, nodo = heapq.heappop(frontera)
        if costo_extraido > mejor_costo.get(nodo.estado, float("inf")):
            repetidos_descartados += 1
            continue
        if problema.es_meta(nodo.estado):
            camino, acciones = reconstruir_camino(nodo)
            return ResultadoBusqueda(
                encontrado=True, camino=camino, acciones=acciones,
                costo=nodo.g, profundidad=nodo.profundidad,
                expandidos=expandidos, generados=generados,
                repetidos_descartados=repetidos_descartados,
                frontera_maxima=frontera_maxima,
                tiempo_ms=(time.perf_counter() - inicio_t) * 1000,
            )
        expandidos += 1
        for accion in problema.acciones(nodo.estado):
            sucesor = problema.resultado(nodo.estado, accion)
            nuevo_costo = nodo.g + problema.costo_paso(
                nodo.estado, accion, sucesor
            )
            generados += 1
            if nuevo_costo < mejor_costo.get(sucesor, float("inf")):
                mejor_costo[sucesor] = nuevo_costo
                hijo = Nodo(
                    estado=sucesor, padre=nodo, accion=accion,
                    g=nuevo_costo, profundidad=nodo.profundidad + 1,
                )
                heapq.heappush(frontera, (nuevo_costo, next(contador), hijo))
                frontera_maxima = max(frontera_maxima, len(frontera))
            else:
                repetidos_descartados += 1

    return ResultadoBusqueda(
        encontrado=False, expandidos=expandidos, generados=generados,
        repetidos_descartados=repetidos_descartados,
        frontera_maxima=frontera_maxima,
        tiempo_ms=(time.perf_counter() - inicio_t) * 1000,
    )

In [ ]:
#Busqueda de profundidad iterativa
def busqueda_profundizacion_iterativa(
    problema: ProblemaLaberinto, limite_maximo: Optional[int] = None
) -> ResultadoBusqueda:
    """
    IDDFS: llama a búsqueda limitada con límites 0, 1, 2, ... Se detiene al encontrar solución, o cuando una vuelta termina SIN
    corte (fracaso definitivo: ya no queda nada más profundo por explorar). `limite_maximo` es una cota de seguridad opcional para
    evitar recursión/ bucle infinito si algo del modelo está mal formulado.
    """
    if limite_maximo is None:
        limite_maximo = len(problema.grafo)  # cota superior segura
 
    expandidos_totales = generados_totales = repetidos_totales = 0
    frontera_maxima_global = 0
    inicio_t = time.perf_counter()
 
    for limite in range(limite_maximo + 1):
        resultado, hubo_corte = busqueda_limitada_profundidad(problema, limite)
        expandidos_totales += resultado.expandidos
        generados_totales += resultado.generados
        repetidos_totales += resultado.repetidos_descartados
        frontera_maxima_global = max(frontera_maxima_global, resultado.frontera_maxima)
        if resultado.encontrado:
            resultado.expandidos = expandidos_totales
            resultado.generados = generados_totales
            resultado.repetidos_descartados = repetidos_totales
            resultado.frontera_maxima = frontera_maxima_global
            resultado.tiempo_ms = (time.perf_counter() - inicio_t) * 1000
            return resultado
        if not hubo_corte:
            # Fracaso definitivo: en la última vuelta no hubo ningun corte por límite, es decir, se agotó el espacio real.
            break
 
    return ResultadoBusqueda(
        encontrado=False, expandidos=expandidos_totales,
        generados=generados_totales,
        repetidos_descartados=repetidos_totales,
        frontera_maxima=frontera_maxima_global,
        tiempo_ms=(time.perf_counter() - inicio_t) * 1000,
    )

In [ ]:
# Busqueda bidireccional (BFS para grafos no dirigidos y costos unitarios)

def busqueda_bidireccional(problema: ProblemaLaberinto) -> ResultadoBusqueda:
    inicio_t = time.perf_counter()
    if problema.inicio == problema.meta:
        return ResultadoBusqueda(
            encontrado=True, camino=[problema.inicio], acciones=[],
            costo=0, profundidad=0, frontera_maxima=1,
            tiempo_ms=(time.perf_counter() - inicio_t) * 1000,
        )

    nodo_ini = Nodo(problema.inicio, None, None, 0, 0)
    nodo_meta = Nodo(problema.meta, None, None, 0, 0)
    frontera_ini = deque([nodo_ini])
    frontera_meta = deque([nodo_meta])
    alcanzados_ini: Dict[Celda, Nodo] = {problema.inicio: nodo_ini}
    alcanzados_meta: Dict[Celda, Nodo] = {problema.meta: nodo_meta}
    expandidos = generados = repetidos_descartados = 0
    frontera_maxima = len(frontera_ini) + len(frontera_meta)

    def expandir_una_capa(frontera, alcanzados_propios, alcanzados_ajenos):
        nonlocal expandidos, generados, repetidos_descartados, frontera_maxima
        cantidad_en_capa = len(frontera)
        for _ in range(cantidad_en_capa):
            nodo = frontera.popleft()
            expandidos += 1
            for accion in problema.acciones(nodo.estado):
                sucesor = problema.resultado(nodo.estado, accion)
                generados += 1
                if sucesor in alcanzados_propios:
                    repetidos_descartados += 1
                    continue
                hijo = Nodo(
                    estado=sucesor, padre=nodo, accion=accion,
                    g=nodo.g + problema.costo_paso(nodo.estado, accion, sucesor),
                    profundidad=nodo.profundidad + 1,
                )
                alcanzados_propios[sucesor] = hijo
                frontera.append(hijo)
                frontera_maxima = max(
                    frontera_maxima, len(frontera_ini) + len(frontera_meta)
                )
                if sucesor in alcanzados_ajenos:
                    return sucesor
        return None

    punto_encuentro = None
    while frontera_ini and frontera_meta and punto_encuentro is None:
        punto_encuentro = expandir_una_capa(
            frontera_ini, alcanzados_ini, alcanzados_meta
        )
        if punto_encuentro is None:
            punto_encuentro = expandir_una_capa(
                frontera_meta, alcanzados_meta, alcanzados_ini
            )

    if punto_encuentro is None:
        return ResultadoBusqueda(
            encontrado=False, expandidos=expandidos, generados=generados,
            repetidos_descartados=repetidos_descartados,
            frontera_maxima=frontera_maxima,
            tiempo_ms=(time.perf_counter() - inicio_t) * 1000,
        )

    camino_desde_inicio, acciones_desde_inicio = reconstruir_camino(
        alcanzados_ini[punto_encuentro]
    )
    camino_desde_meta, _ = reconstruir_camino(alcanzados_meta[punto_encuentro])
    camino_hacia_meta = list(reversed(camino_desde_meta))
    camino_completo = camino_desde_inicio + camino_hacia_meta[1:]
    acciones_completas = acciones_desde_inicio + camino_hacia_meta[1:]
    costo_total = sum(
        problema.costo_paso(origen, destino, destino)
        for origen, destino in zip(camino_completo, camino_completo[1:])
    )
    return ResultadoBusqueda(
        encontrado=True, camino=camino_completo, acciones=acciones_completas,
        costo=costo_total, profundidad=len(camino_completo) - 1,
        expandidos=expandidos, generados=generados,
        repetidos_descartados=repetidos_descartados,
        frontera_maxima=frontera_maxima,
        tiempo_ms=(time.perf_counter() - inicio_t) * 1000,
    )

In [ ]:
def busqueda_limitada_profundidad(
    problema: ProblemaLaberinto, limite: int
) -> Tuple[ResultadoBusqueda, bool]:
    """
    DFS con límite de profundidad. Devuelve (resultado, hubo_corte). `hubo_corte=True` significa "no se encontró solución DENTRO del
    límite, pero podría existir más profundo" (CORTE, distinto de FRACASO definitivo, que sería `hubo_corte=False` con
    `encontrado=False`: en ese caso se agotó todo el espacio alcanzable sin necesidad de cortar por profundidad).
    """
    inicio_t = time.perf_counter()
    expandidos = generados = repetidos_descartados = 0
    frontera_maxima = 0
    hubo_corte = False
 
    """
    Pila explícita con marca de expansión SOLO en la rama actual(no global), porque en DFS limitado por profundidad conviene
    permitir revisitar un estado por otra rama si el límite lo permite; para simplificar y mantener el contrato de "búsqueda en
    grafo" se usa aquí una marca por profundidad mínima alcanzada.
    """
    mejor_profundidad_vista: Dict[Celda, int] = {}

    nodo_inicial = Nodo(estado=problema.inicio, padre=None, accion=None, g=0, profundidad=0)
    pila = [nodo_inicial]
    frontera_maxima = len(pila)
 
    while pila:
        frontera_maxima = max(frontera_maxima, len(pila))
        nodo = pila.pop()
        if problema.es_meta(nodo.estado):
            camino, acciones = reconstruir_camino(nodo)
            resultado = ResultadoBusqueda(
                encontrado=True, camino=camino, acciones=acciones,
                costo=nodo.g, profundidad=nodo.profundidad,
                expandidos=expandidos, generados=generados,
                repetidos_descartados=repetidos_descartados,
                frontera_maxima=frontera_maxima,
                tiempo_ms=(time.perf_counter() - inicio_t) * 1000,
            )
            return resultado, False
        if nodo.profundidad >= limite:
            hubo_corte = True
            continue
        if (nodo.estado in mejor_profundidad_vista and
                mejor_profundidad_vista[nodo.estado] <= nodo.profundidad):
            repetidos_descartados += 1
            continue
        mejor_profundidad_vista[nodo.estado] = nodo.profundidad
        expandidos += 1
        for accion in problema.acciones(nodo.estado):
            sucesor = problema.resultado(nodo.estado, accion)
            generados += 1
            hijo = Nodo(
                estado=sucesor, padre=nodo, accion=accion,
                g=nodo.g + problema.costo_paso(nodo.estado, accion, sucesor),
                profundidad=nodo.profundidad + 1,
            )
            pila.append(hijo)
 
    resultado = ResultadoBusqueda(
        encontrado=False, expandidos=expandidos, generados=generados,
        repetidos_descartados=repetidos_descartados,
        frontera_maxima=frontera_maxima,
        tiempo_ms=(time.perf_counter() - inicio_t) * 1000,
    )
    return resultado, hubo_corte

## 7. Lee: propagación de un frente luminoso

No se modela un único rayo que rebota. Se modela una onda que avanza simultáneamente por todos los corredores accesibles.

```text
LEE(grafo, inicio, meta):
    etiqueta[inicio] ← 0
    frente_actual ← {inicio}

    mientras frente_actual no esté vacío y meta no tenga etiqueta:
        frente_siguiente ← vacío
        para cada celda del frente_actual:
            para cada vecino transitable sin etiqueta:
                etiqueta[vecino] ← etiqueta[celda] + 1
                registrar procedencia o dirección
                agregar vecino al frente_siguiente
        frente_actual ← frente_siguiente

    si meta no tiene etiqueta: devolver fracaso
    reconstruir desde meta siguiendo etiquetas decrecientes
```

### Requisitos adicionales

- Dibuje al menos ocho instantes del frente de onda.
- Use una escala de color que represente el tiempo de llegada.
- Compruebe que la etiqueta de la meta coincide con la profundidad de BFS.
- Explique formalmente por qué Lee es BFS en un grafo no ponderado.
- Analice qué deja de funcionar cuando los costos no son unitarios.

### ¿Por qué Lee es BFS?

En un grafo no ponderado, Lee comienza asignando distancia cero al inicio y expande por capas. Sea $L_k$ el conjunto de estados etiquetados con $k$. La capa inicial es $L_0 = \{inicio\}$. Si todo estado de $L_k$ tiene distancia mínima $k$, cada vecino no etiquetado recibe $k+1$ y pertenece a $L_{k+1}$. Como las capas se procesan en orden creciente, ningún camino más corto hacia ese vecino puede quedar sin procesar. Por inducción, la etiqueta de cada estado es su distancia mínima desde el inicio.

Así, el primer nivel que alcanza la meta tiene el menor número posible de aristas. Esto es exactamente la propiedad de optimalidad de BFS con costos unitarios. El historial de frentes es evidencia adicional para visualizar la expansión; no participa en la decisión del algoritmo.

En un grafo con $|V|$ vértices y $|E|$ aristas, una implementación con conjunto de alcanzados cuesta $O(|V|+|E|)$ tiempo y $O(|V|)$ espacio. En notación de árbol de búsqueda, suele expresarse como $O(b^{d+1})$ tiempo y espacio, donde $b$ es el factor de ramificación y $d$ la profundidad de la solución más cercana.

### ¿Qué deja de ser válido con costos distintos de uno?

Las etiquetas de Lee cuentan aristas, no costo acumulado. Si los pesos varían, la ruta con menos pasos puede ser más costosa que otra más larga. Lee sigue encontrando una ruta con el mínimo número de pasos, pero ya no garantiza costo mínimo. Para pesos positivos debe usarse UCS o una variante ponderada apropiada.

### SimpleAI y AIMA-Python

En SimpleAI, el adaptador expresa `actions`, `result`, `is_goal` y `cost`; el algoritmo de la biblioteca recibe `graph_search=True` para evitar ciclos. En AIMA-Python, el adaptador define `actions`, `result`, `goal_test` y `path_cost`. El costo de AIMA se acumula con la forma `c + costo_de_transicion`.

La instrumentación por llamadas a `actions` permite contar cuántas veces la biblioteca pidió sucesores y cuántas acciones legales se devolvieron. No revela necesariamente cuántos nodos aceptó en la frontera, cuántos repetidos descartó ni el tamaño máximo de la frontera. Esas métricas deben reportarse como no disponibles si no se instrumenta el interior del algoritmo. No se deben equiparar las llamadas a `actions` con nodos generados sin explicar la convención.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle


def lee_frente_de_onda(problema: ProblemaLaberinto):
    """Expande por capas y conserva las etiquetas y frentes de Lee."""
    inicio_t = time.perf_counter()
    etiqueta: Dict[Celda, int] = {problema.inicio: 0}
    procedencia: Dict[Celda, Celda] = {}
    frente_actual = [problema.inicio]
    historial_frentes: List[List[Celda]] = [list(frente_actual)]

    expandidos = generados = repetidos_descartados = 0
    frontera_maxima = len(frente_actual)

    while frente_actual and problema.meta not in etiqueta:
        frontera_maxima = max(frontera_maxima, len(frente_actual))
        frente_siguiente: List[Celda] = []
        for celda in frente_actual:
            expandidos += 1
            for vecino in problema.acciones(celda):
                generados += 1
                if vecino in etiqueta:
                    repetidos_descartados += 1
                    continue
                etiqueta[vecino] = etiqueta[celda] + 1
                procedencia[vecino] = celda
                frente_siguiente.append(vecino)
        frontera_maxima = max(frontera_maxima, len(frente_siguiente))
        frente_actual = frente_siguiente
        if frente_actual:
            historial_frentes.append(list(frente_actual))

    tiempo_ms = (time.perf_counter() - inicio_t) * 1000

    if problema.meta not in etiqueta:
        resultado = ResultadoBusqueda(
            encontrado=False, expandidos=expandidos, generados=generados,
            repetidos_descartados=repetidos_descartados,
            frontera_maxima=frontera_maxima, tiempo_ms=tiempo_ms,
        )
        return resultado, etiqueta, historial_frentes

    camino = [problema.meta]
    while camino[-1] != problema.inicio:
        camino.append(procedencia[camino[-1]])
    camino.reverse()
    acciones = camino[1:]

    resultado = ResultadoBusqueda(
        encontrado=True, camino=camino, acciones=acciones,
        costo=etiqueta[problema.meta], profundidad=etiqueta[problema.meta],
        expandidos=expandidos, generados=generados,
        repetidos_descartados=repetidos_descartados,
        frontera_maxima=frontera_maxima, tiempo_ms=tiempo_ms,
    )
    return resultado, etiqueta, historial_frentes


def _dibujar_paredes_laberinto(ax, grafo: Grafo, filas: int, columnas: int,
                                color_pared="black", grosor=1.6) -> None:
    """Dibuja los bordes y todas las paredes internas que no son corredores."""
    ax.set_xlim(0, columnas)
    ax.set_ylim(0, filas)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    ax.axis("off")

    ax.plot([0, columnas], [0, 0], color=color_pared, linewidth=grosor)
    ax.plot([0, columnas], [filas, filas], color=color_pared, linewidth=grosor)
    ax.plot([0, 0], [0, filas], color=color_pared, linewidth=grosor)
    ax.plot([columnas, columnas], [0, filas], color=color_pared, linewidth=grosor)

    for fila in range(filas):
        for columna in range(columnas):
            celda = (fila, columna)
            if columna + 1 < columnas and (fila, columna + 1) not in grafo[celda]:
                ax.plot([columna + 1, columna + 1], [fila, fila + 1],
                        color=color_pared, linewidth=grosor)
            if fila + 1 < filas and (fila + 1, columna) not in grafo[celda]:
                ax.plot([columna, columna + 1], [fila + 1, fila + 1],
                        color=color_pared, linewidth=grosor)


def _rellenar_tiempos_de_llegada(ax, etiquetas_acumuladas, normalizador, mapa_color):
    for (fila, columna), tiempo_llegada in etiquetas_acumuladas.items():
        ax.add_patch(Rectangle(
            (columna, fila), 1, 1,
            facecolor=mapa_color(normalizador(tiempo_llegada)),
            edgecolor="none", zorder=0,
        ))


def _marcar_inicio_y_meta(ax, inicio: Celda, meta: Celda) -> None:
    ax.add_patch(Rectangle((inicio[1], inicio[0]), 1, 1, facecolor="none",
                           edgecolor="#1b8a3a", linewidth=2.2, zorder=3))
    ax.text(inicio[1] + 0.5, inicio[0] + 0.5, "I", color="#1b8a3a", fontsize=8,
            fontweight="bold", ha="center", va="center", zorder=4)
    ax.add_patch(Rectangle((meta[1], meta[0]), 1, 1, facecolor="none",
                           edgecolor="#c0392b", linewidth=2.2, zorder=3))
    ax.text(meta[1] + 0.5, meta[0] + 0.5, "M", color="#c0392b", fontsize=8,
            fontweight="bold", ha="center", va="center", zorder=4)


def graficar_frentes_lee(historial_frentes, grafo, filas, columnas, inicio,
                          meta=None, maximo_instantes=8):
    """Muestra hasta ocho frentes sobre el laberinto recibido como argumento."""
    if not historial_frentes:
        raise ValueError("El historial de frentes esta vacio")
    if maximo_instantes < 1:
        raise ValueError("maximo_instantes debe ser positivo")
    total_frentes = len(historial_frentes)
    if total_frentes <= maximo_instantes:
        indices_a_mostrar = list(range(total_frentes))
    elif maximo_instantes == 1:
        indices_a_mostrar = [total_frentes - 1]
    else:
        paso = (total_frentes - 1) / (maximo_instantes - 1)
        indices_a_mostrar = sorted({round(i * paso) for i in range(maximo_instantes)})

    numero = len(indices_a_mostrar)
    columnas_fig = min(4, numero)
    filas_fig = (numero + columnas_fig - 1) // columnas_fig
    fig, ejes = plt.subplots(filas_fig, columnas_fig,
                              figsize=(4 * columnas_fig, 4 * filas_fig))
    ejes = [ejes] if numero == 1 else list(ejes.flat)
    normalizador = Normalize(vmin=0, vmax=max(1, total_frentes - 1))
    mapa_color = plt.cm.plasma

    for posicion, indice_frente in enumerate(indices_a_mostrar):
        ax = ejes[posicion]
        etiquetas_acumuladas = {
            celda: tiempo_llegada
            for tiempo_llegada, frente in enumerate(historial_frentes[:indice_frente + 1])
            for celda in frente
        }
        _rellenar_tiempos_de_llegada(ax, etiquetas_acumuladas, normalizador, mapa_color)
        _dibujar_paredes_laberinto(ax, grafo, filas, columnas)
        if meta is not None:
            _marcar_inicio_y_meta(ax, inicio, meta)
        ax.set_title(f"Instante t={indice_frente}")

    for posicion in range(numero, len(ejes)):
        ejes[posicion].axis("off")
    fig.suptitle("Propagacion del frente de onda (Lee)")
    fig.colorbar(
        plt.cm.ScalarMappable(norm=normalizador, cmap=mapa_color),
        ax=ejes, shrink=0.6, label="Tiempo de llegada (frentes)",
    )
    fig.tight_layout()
    plt.show()


def graficar_laberinto_con_camino(grafo: Grafo, filas: int, columnas: int,
                                   inicio: Celda, meta: Celda,
                                   camino: List[Celda]) -> None:
    fig, ax = plt.subplots(figsize=(6, 6))
    _dibujar_paredes_laberinto(ax, grafo, filas, columnas)
    if camino:
        xs = [columna + 0.5 for _, columna in camino]
        ys = [fila + 0.5 for fila, _ in camino]
        ax.plot(xs, ys, color="#1b6fc0", linewidth=3.0, zorder=2,
                solid_capstyle="round")
    _marcar_inicio_y_meta(ax, inicio, meta)
    ax.set_title(f"Laberinto {filas}x{columnas}: camino de Lee/BFS")
    plt.show()


problema_lee = ProblemaLaberinto(grafo, inicio, meta)
resultado_lee, etiquetas_lee, historial_frentes = lee_frente_de_onda(problema_lee)
print(f"Lee -> encontrado={resultado_lee.encontrado}  "
      f"etiqueta_meta={etiquetas_lee.get(meta)}  "
      f"pasos={resultado_lee.profundidad}  "
      f"tiempo_ms={resultado_lee.tiempo_ms:.3f}")
resultado_bfs_verificacion = busqueda_bfs(problema_lee)
assert etiquetas_lee[meta] == resultado_bfs_verificacion.profundidad
print(f"Verificacion OK: Lee y BFS = {etiquetas_lee[meta]} pasos")
graficar_frentes_lee(
    historial_frentes, grafo, filas, columnas, inicio,
    meta=meta, maximo_instantes=8,
)
graficar_laberinto_con_camino(
    grafo, filas, columnas, inicio, meta, resultado_lee.camino,
)

# Versión 2 — SimpleAI

Instalación orientativa:

```text
pip install simpleai
```

Consulte la documentación oficial. Debe crear una subclase de `SearchProblem` y definir, como mínimo:

| Método de SimpleAI | Responsabilidad en el laberinto |
|---|---|
| `actions(state)` | Producir únicamente movimientos legales |
| `result(state, action)` | Calcular el estado sucesor sin mutar el original |
| `is_goal(state)` | Reconocer la celda meta |
| `cost(state, action, state2)` | Devolver el costo del corredor |

Instancie el problema con un estado inicial inmutable y use `graph_search=True`.

Debe ejecutar y estudiar:

- `depth_first`;
- `breadth_first`;
- `uniform_cost`;
- `limited_depth_first`;
- `iterative_limited_depth_first`.

### Exigencia avanzada

SimpleAI devuelve un nodo solución, pero las métricas del contrato no aparecen automáticamente con la misma definición. Investigue su mecanismo de *viewer* o diseñe instrumentación externa **sin modificar el código fuente instalado de la biblioteca**. Documente cualquier diferencia entre las métricas de SimpleAI y las de su versión.

No copie la estructura de su buscador desde cero dentro de esta versión: el propósito es construir correctamente el adaptador del problema y comprender la biblioteca.

In [ ]:
# %pip install simpleai aima3


In [ ]:
# Version 2: adaptador SimpleAI. La biblioteca no dibuja resultados por si misma;
# se visualizan las rutas normalizadas despues de ejecutar sus algoritmos.
from collections import deque
from dataclasses import dataclass
import csv
import importlib
import math
import random
import statistics
import time
from typing import Any, Callable, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple


@dataclass
class ResultadoAdaptador:
    """Resultado normalizado; las metricas internas no expuestas quedan en None."""
    encontrado: bool
    camino: List[Any]
    acciones: List[Any]
    costo: float
    profundidad: int
    expandidos: Optional[int]
    generados: Optional[int]
    repetidos_descartados: Optional[int]
    frontera_maxima: Optional[int]
    tiempo_ms: float
    llamadas_acciones: Optional[int] = None


@dataclass
class InstrumentacionBiblioteca:
    llamadas_acciones: int = 0
    sucesores_considerados: int = 0
    llamadas_costo: int = 0


def _crear_resultado_desde_nodo(nodo, problema, instrumentacion):
    if nodo is None or nodo is False or isinstance(nodo, str):
        return ResultadoAdaptador(
            encontrado=False, camino=[], acciones=[], costo=math.inf,
            profundidad=0,
            expandidos=instrumentacion.llamadas_acciones,
            generados=instrumentacion.sucesores_considerados,
            repetidos_descartados=None, frontera_maxima=None,
            tiempo_ms=0.0,
            llamadas_acciones=instrumentacion.llamadas_acciones,
        )
    nodos = []
    actual = nodo
    while actual is not None:
        nodos.append(actual)
        actual = getattr(actual, "parent", None)
    nodos.reverse()
    camino = [nodo_busqueda.state for nodo_busqueda in nodos]
    acciones = [nodo_busqueda.action for nodo_busqueda in nodos[1:]]
    costo = sum(
        problema.costo_paso(origen, destino, destino)
        for origen, destino in zip(camino, camino[1:])
    )
    return ResultadoAdaptador(
        encontrado=bool(camino) and camino[-1] == problema.meta,
        camino=camino, acciones=acciones, costo=costo,
        profundidad=max(0, len(camino) - 1),
        expandidos=instrumentacion.llamadas_acciones,
        generados=instrumentacion.sucesores_considerados,
        repetidos_descartados=None, frontera_maxima=None, tiempo_ms=0.0,
        llamadas_acciones=instrumentacion.llamadas_acciones,
    )


def _resultado_con_tiempo(nodo, problema, instrumentacion, inicio_reloj):
    resultado = _crear_resultado_desde_nodo(nodo, problema, instrumentacion)
    resultado.tiempo_ms = (time.perf_counter() - inicio_reloj) * 1000
    return resultado


def ejecutar_simpleai(problema, algoritmo: str, limite: Optional[int] = None):
    """Ejecuta SimpleAI y normaliza el nodo solucion al contrato del taller.

    `expandidos` es el numero de llamadas a `actions`; `generados` cuenta las
    acciones devueltas por esas llamadas. SimpleAI no expone de forma uniforme
    repetidos descartados ni frontera maxima, por eso esas metricas son None.
    """
    try:
        modulo_simpleai = importlib.import_module("simpleai.search")
    except ImportError as error:
        raise ImportError(
            "Instala SimpleAI en este kernel con %pip install simpleai"
        ) from error
    instrumentacion = InstrumentacionBiblioteca()

    class ProblemaLaberintoSimpleAI(modulo_simpleai.SearchProblem):
        def __init__(self):
            super().__init__(problema.inicio)

        def actions(self, state):
            instrumentacion.llamadas_acciones += 1
            acciones = sorted(problema.grafo[state])
            instrumentacion.sucesores_considerados += len(acciones)
            return acciones

        def result(self, state, action):
            return action

        def is_goal(self, state):
            return state == problema.meta

        def cost(self, state, action, state2):
            instrumentacion.llamadas_costo += 1
            return problema.costo_paso(state, action, state2)

    instancia = ProblemaLaberintoSimpleAI()
    inicio_reloj = time.perf_counter()
    if algoritmo == "DFS":
        nodo = modulo_simpleai.depth_first(instancia, graph_search=True)
    elif algoritmo == "BFS":
        nodo = modulo_simpleai.breadth_first(instancia, graph_search=True)
    elif algoritmo == "UCS":
        nodo = modulo_simpleai.uniform_cost(instancia, graph_search=True)
    elif algoritmo == "DLS":
        limite = len(problema.grafo) if limite is None else limite
        nodo = modulo_simpleai.limited_depth_first(
            instancia, limite, graph_search=True
        )
    elif algoritmo == "IDDFS":
        nodo = modulo_simpleai.iterative_limited_depth_first(
            instancia, graph_search=True
        )
    else:
        raise ValueError(f"Algoritmo SimpleAI desconocido: {algoritmo}")
    return _resultado_con_tiempo(nodo, problema, instrumentacion, inicio_reloj)


def graficar_caminos_versiones(grafo, filas, columnas, inicio, meta,
                                resultados, titulo):
    """Dibuja en paneles separados las rutas de cada algoritmo ejecutado."""
    import matplotlib.pyplot as plt

    encontrados = [
        (nombre, resultado) for nombre, resultado in resultados.items()
        if resultado.encontrado
    ]
    if not encontrados:
        print(f"Sin rutas disponibles para graficar: {titulo}")
        return
    fig, ejes = plt.subplots(
        1, len(encontrados), figsize=(5 * len(encontrados), 5), squeeze=False
    )
    for ax, (nombre, resultado) in zip(ejes.flat, encontrados):
        _dibujar_paredes_laberinto(ax, grafo, filas, columnas)
        xs = [columna + 0.5 for _, columna in resultado.camino]
        ys = [fila + 0.5 for fila, _ in resultado.camino]
        ax.plot(xs, ys, linewidth=2.5, color="#1769aa", zorder=2)
        _marcar_inicio_y_meta(ax, inicio, meta)
        ax.set_title(f"{nombre}: {resultado.profundidad} pasos")
    fig.suptitle(titulo)
    fig.tight_layout()
    plt.show()


resultados_simpleai = {}
try:
    importlib.import_module("simpleai.search")
except ImportError:
    print("SimpleAI no esta instalado. Instala simpleai y vuelve a ejecutar esta celda.")
else:
    for nombre in ("DFS", "BFS", "UCS", "DLS", "IDDFS"):
        resultados_simpleai[nombre] = ejecutar_simpleai(problema_lee, nombre)
    for nombre, resultado in resultados_simpleai.items():
        print(
            f"SimpleAI/{nombre}: encontrado={resultado.encontrado}, "
            f"pasos={resultado.profundidad}, costo={resultado.costo:g}, "
            f"llamadas_actions={resultado.llamadas_acciones}, "
            f"tiempo_ms={resultado.tiempo_ms:.3f}"
        )
    graficar_caminos_versiones(
        grafo, filas, columnas, inicio, meta, resultados_simpleai,
        "Rutas de los algoritmos SimpleAI",
    )

# Versión 3 — AIMA-Python

Repositorio de referencia: `aimacode/aima-python`.

Debe modelar el laberinto como una subclase de `Problem`. Investigue y documente los contratos de:

| Método de AIMA-Python | Decisión requerida |
|---|---|
| `actions(state)` | Formato de las acciones y orden determinista |
| `result(state, action)` | Transición entre celdas |
| `goal_test(state)` | Prueba de objetivo |
| `path_cost(c, state1, action, state2)` | Acumulación de costos |

Compare al menos:

- `depth_first_graph_search`;
- `breadth_first_graph_search`;
- `uniform_cost_search`;
- `depth_limited_search`;
- `iterative_deepening_search`;
- `bidirectional_search`.

AIMA-Python no está disponible en los canales Conda configurados para este kernel. Usa la copia del repositorio indicada por el curso y agrega a `sys.path` la carpeta que contiene el módulo `search` (`aima/search.py` o `search.py`). Después vuelve a ejecutar la celda de código de esta versión. El adaptador prueba esos nombres de módulo y muestra un aviso si no encuentra ninguno.

### Exigencia avanzada

Instrumenta los algoritmos sin alterar permanentemente el repositorio. Puede utilizar una subclase del problema, envoltorios o contadores cuidadosamente definidos. Explique por qué contar llamadas a `actions` no siempre coincide con contar nodos generados.

Compare la representación del nodo de AIMA-Python con su propia clase de nodo. La celda de esta versión dibuja las rutas de cada algoritmo cuando el módulo está disponible.

In [ ]:
# Version 3: adaptador AIMA-Python oficial y compatibilidad controlada con aima3.
import inspect


def _importar_modulo_busqueda_aima():
    errores = []
    for nombre_modulo in ("aima.search", "aima3.search", "search"):
        try:
            return importlib.import_module(nombre_modulo)
        except ImportError as error:
            errores.append(error)
    raise ImportError(
        "No se encontro search.py de AIMA-Python. Agrega el repositorio del curso "
        "a sys.path o instala la distribucion correspondiente."
    ) from errores[-1]


def _resolver_funcion_aima(modulo, *nombres):
    for nombre in nombres:
        funcion = getattr(modulo, nombre, None)
        if callable(funcion):
            return funcion
    disponibles = [nombre for nombre in dir(modulo) if "search" in nombre.lower()]
    raise AttributeError(
        f"AIMA no expone ninguna de {nombres}. Funciones detectadas: {disponibles}"
    )


def _llamar_busqueda_aima(funcion, problema, limite=None):
    if limite is None:
        return funcion(problema)
    parametros = inspect.signature(funcion).parameters
    if "limit" in parametros:
        return funcion(problema, limit=limite)
    if "depth_limit" in parametros:
        return funcion(problema, depth_limit=limite)
    if "depth" in parametros:
        return funcion(problema, depth=limite)
    return funcion(problema, limite)


def ejecutar_aima(problema, algoritmo: str, limite: Optional[int] = None):
    """Ejecuta AIMA con adaptacion de estados para las busquedas limitadas.

    aima3.search usa `breadth_first_search` y sus DLS/IDDFS no detectan ciclos.
    Para estos dos algoritmos el estado lleva su profundidad y se podan rutas
    dominadas dentro de cada iteracion. El limite DLS por defecto es la
    profundidad de BFS, no el numero total de celdas.
    """
    modulo = _importar_modulo_busqueda_aima()
    instrumentacion = InstrumentacionBiblioteca()
    codificar_profundidad = algoritmo in ("DLS", "IDDFS")
    mejor_profundidad = {}

    class ProblemaLaberintoAIMA(modulo.Problem):
        def __init__(self):
            estado_inicial = (problema.inicio, 0) if codificar_profundidad else problema.inicio
            super().__init__(estado_inicial, problema.meta)

        def actions(self, state):
            instrumentacion.llamadas_acciones += 1
            if codificar_profundidad:
                celda, profundidad = state
                if profundidad == 0:
                    mejor_profundidad.clear()
                anterior = mejor_profundidad.get(celda)
                if anterior is not None and anterior <= profundidad:
                    return []
                mejor_profundidad[celda] = profundidad
            else:
                celda = state
            acciones = sorted(problema.grafo[celda])
            instrumentacion.sucesores_considerados += len(acciones)
            return acciones

        def result(self, state, action):
            if codificar_profundidad:
                _, profundidad = state
                return action, profundidad + 1
            return action

        def goal_test(self, state):
            celda = state[0] if codificar_profundidad else state
            return celda == problema.meta

        def path_cost(self, cost_so_far, state1, action, state2):
            instrumentacion.llamadas_costo += 1
            origen = state1[0] if codificar_profundidad else state1
            destino = state2[0] if codificar_profundidad else state2
            return cost_so_far + problema.costo_paso(origen, action, destino)

    instancia = ProblemaLaberintoAIMA()
    inicio_reloj = time.perf_counter()
    if algoritmo == "DFS":
        funcion = _resolver_funcion_aima(
            modulo, "depth_first_graph_search", "depth_first_search"
        )
        nodo = _llamar_busqueda_aima(funcion, instancia)
    elif algoritmo == "BFS":
        funcion = _resolver_funcion_aima(
            modulo, "breadth_first_graph_search", "breadth_first_search"
        )
        nodo = _llamar_busqueda_aima(funcion, instancia)
    elif algoritmo == "UCS":
        funcion = _resolver_funcion_aima(
            modulo, "uniform_cost_search", "uniform_cost_graph_search"
        )
        nodo = _llamar_busqueda_aima(funcion, instancia)
    elif algoritmo == "DLS":
        funcion = _resolver_funcion_aima(
            modulo, "depth_limited_search", "depth_limited_graph_search"
        )
        limite = busqueda_bfs(problema).profundidad if limite is None else limite
        nodo = _llamar_busqueda_aima(funcion, instancia, limite)
    elif algoritmo == "IDDFS":
        funcion = _resolver_funcion_aima(
            modulo, "iterative_deepening_search", "iterative_deepening_graph_search"
        )
        nodo = _llamar_busqueda_aima(funcion, instancia)
    elif algoritmo == "BIDIRECCIONAL":
        if modulo.__name__.startswith("aima3."):
            raise NotImplementedError(
                "aima3.search.bidirectional_search devuelve solo un costo y requiere "
                "metodos adicionales; para obtener camino use el repositorio AIMA oficial."
            )
        funcion = _resolver_funcion_aima(
            modulo, "bidirectional_search", "bidirectional_graph_search"
        )
        nodo = _llamar_busqueda_aima(funcion, instancia)
    else:
        raise ValueError(f"Algoritmo AIMA-Python desconocido: {algoritmo}")

    def extraer_estado(estado):
        return estado[0] if codificar_profundidad else estado

    nodos = []
    actual = nodo
    if nodo is None or isinstance(nodo, str):
        resultado = ResultadoAdaptador(
            encontrado=False, camino=[], acciones=[], costo=math.inf,
            profundidad=0, expandidos=instrumentacion.llamadas_acciones,
            generados=instrumentacion.sucesores_considerados,
            repetidos_descartados=None, frontera_maxima=None, tiempo_ms=0.0,
            llamadas_acciones=instrumentacion.llamadas_acciones,
        )
    else:
        while actual is not None:
            nodos.append(actual)
            actual = getattr(actual, "parent", None)
        nodos.reverse()
        camino = [extraer_estado(nodo_busqueda.state) for nodo_busqueda in nodos]
        acciones = [nodo_busqueda.action for nodo_busqueda in nodos[1:]]
        costo = sum(
            problema.costo_paso(origen, destino, destino)
            for origen, destino in zip(camino, camino[1:])
        )
        resultado = ResultadoAdaptador(
            encontrado=bool(camino) and camino[-1] == problema.meta,
            camino=camino, acciones=acciones, costo=costo,
            profundidad=max(0, len(camino) - 1),
            expandidos=instrumentacion.llamadas_acciones,
            generados=instrumentacion.sucesores_considerados,
            repetidos_descartados=None, frontera_maxima=None, tiempo_ms=0.0,
            llamadas_acciones=instrumentacion.llamadas_acciones,
        )
    resultado.tiempo_ms = (time.perf_counter() - inicio_reloj) * 1000
    return resultado


resultados_aima = {}
try:
    modulo_aima = _importar_modulo_busqueda_aima()
except ImportError as error:
    print(f"AIMA-Python no esta disponible: {error}")
else:
    print("API AIMA detectada:", modulo_aima.__name__)
    for nombre in ("DFS", "BFS", "UCS", "DLS", "IDDFS", "BIDIRECCIONAL"):
        try:
            resultados_aima[nombre] = ejecutar_aima(problema_lee, nombre)
        except NotImplementedError as error:
            print(f"AIMA/{nombre} omitida: {error}")
    for nombre, resultado in resultados_aima.items():
        print(
            f"AIMA-Python/{nombre}: encontrado={resultado.encontrado}, "
            f"pasos={resultado.profundidad}, costo={resultado.costo:g}, "
            f"llamadas_actions={resultado.llamadas_acciones}, "
            f"tiempo_ms={resultado.tiempo_ms:.3f}"
        )
    graficar_caminos_versiones(
        grafo, filas, columnas, inicio, meta, resultados_aima,
        "Rutas de los algoritmos AIMA-Python",
    )

## 8. Pruebas de aceptación

El estudiante debe implementar pruebas automáticas equivalentes a las siguientes especificaciones, sin recibir su solución:

### Validez de un camino

Para cada resultado exitoso:

1. el primer estado es el inicio;
2. el último estado es la meta;
3. cada pareja consecutiva aparece como arista en el grafo;
4. el costo reportado coincide con la suma de costos;
5. aplicar las acciones reproduce exactamente los estados.

### Concordancia entre versiones

- En costos unitarios: `longitud(BFS) = longitud(UCS) = etiqueta_meta(Lee)`.
- En un árbol: todas las soluciones válidas contienen la misma secuencia de estados.
- Con ciclos: `longitud(BFS) ≤ longitud(DFS)` para una DFS que encuentre solución.
- Con costos positivos: `costo(UCS) ≤ costo(cualquier solución BFS)`.
- Las tres versiones deben concordar en existencia de solución y costo óptimo.

### Casos límite obligatorios

- inicio igual a meta;
- laberinto de `1×1`;
- meta inalcanzable después de eliminar corredores;
- múltiples entradas obsoletas en UCS;
- dos posibles puntos de encuentro bidireccional;
- límite exacto, insuficiente y excesivo en búsqueda limitada.

In [ ]:
import math
import unittest


def verificar_camino_valido(problema: ProblemaLaberinto, resultado) -> None:
    """Verifica extremos, acciones, aristas y costo acumulado del camino."""
    if not resultado.encontrado:
        assert resultado.camino == [], "camino debe quedar vacio sin solucion"
        return
    camino = resultado.camino
    assert camino and camino[0] == problema.inicio
    assert camino[-1] == problema.meta
    assert len(resultado.acciones) == len(camino) - 1
    for origen, destino in zip(camino, camino[1:]):
        assert destino in problema.grafo[origen]
    estado = problema.inicio
    estados_reconstruidos = [estado]
    for accion in resultado.acciones:
        estado = problema.resultado(estado, accion)
        estados_reconstruidos.append(estado)
    assert estados_reconstruidos == camino
    costo = sum(
        problema.costo_paso(origen, destino, destino)
        for origen, destino in zip(camino, camino[1:])
    )
    assert math.isclose(costo, resultado.costo, rel_tol=1e-9, abs_tol=1e-9)


class TestAceptacion(unittest.TestCase):
    def setUp(self):
        self.problema = ProblemaLaberinto(grafo, inicio, meta)

    def test_caminos_validos_algoritmos_desde_cero(self):
        solucionadores = (
            busqueda_dfs, busqueda_bfs, busqueda_costo_uniforme,
            busqueda_bidireccional, busqueda_profundizacion_iterativa,
            lambda problema: lee_frente_de_onda(problema)[0],
        )
        for resolver in solucionadores:
            with self.subTest(resolver=resolver):
                resultado = resolver(self.problema)
                self.assertTrue(resultado.encontrado)
                verificar_camino_valido(self.problema, resultado)

    def test_bfs_ucs_lee_coinciden_en_arbol_unitario(self):
        bfs = busqueda_bfs(self.problema)
        ucs = busqueda_costo_uniforme(self.problema)
        lee, etiquetas, _ = lee_frente_de_onda(self.problema)
        self.assertEqual(bfs.profundidad, ucs.profundidad)
        self.assertEqual(bfs.profundidad, etiquetas[meta])
        self.assertEqual(bfs.camino, ucs.camino)
        self.assertEqual(bfs.camino, lee.camino)

    def test_dfs_en_ciclos_encuentra_ruta_valida(self):
        grafo_ciclico, _, _ = agregar_ciclos(
            grafo, filas, columnas, semilla, fraccion=0.05
        )
        problema_ciclico = ProblemaLaberinto(grafo_ciclico, inicio, meta)
        resultado_bfs = busqueda_bfs(problema_ciclico)
        resultado_dfs = busqueda_dfs(problema_ciclico)
        self.assertLessEqual(resultado_bfs.profundidad, resultado_dfs.profundidad)
        verificar_camino_valido(problema_ciclico, resultado_bfs)
        verificar_camino_valido(problema_ciclico, resultado_dfs)

    def test_ucs_es_mas_barato_que_bfs_en_caso_ponderado(self):
        resultado_bfs, resultado_ucs = probar_divergencia_bfs_ucs(
            ProblemaLaberinto, busqueda_bfs, busqueda_costo_uniforme
        )
        self.assertEqual((resultado_bfs.profundidad, resultado_bfs.costo), (2, 20))
        self.assertEqual((resultado_ucs.profundidad, resultado_ucs.costo), (3, 3))

    def test_ucs_descarta_entradas_obsoletas(self):
        resultado = probar_ucs_entradas_obsoletas(
            busqueda_costo_uniforme, ProblemaLaberinto
        )
        self.assertEqual(resultado.camino, ["A", "F", "D", "G"])
        self.assertGreaterEqual(resultado.repetidos_descartados, 2)

    def test_inicio_igual_a_meta(self):
        problema = ProblemaLaberinto(grafo, inicio, inicio)
        for resolver in (busqueda_dfs, busqueda_bfs, busqueda_costo_uniforme,
                         busqueda_bidireccional):
            resultado = resolver(problema)
            self.assertTrue(resultado.encontrado)
            self.assertEqual(resultado.camino, [inicio])
            self.assertEqual(resultado.costo, 0)
            self.assertEqual(resultado.expandidos, 0)

    def test_laberinto_1x1(self):
        grafo_1x1 = LaberintoHuntKill(1, 1, 1).generar()
        problema = ProblemaLaberinto(grafo_1x1, (0, 0), (0, 0))
        resultado = busqueda_bfs(problema)
        self.assertTrue(resultado.encontrado)
        self.assertEqual(resultado.camino, [(0, 0)])

    def test_meta_inalcanzable(self):
        grafo_roto = {estado: set(vecinos) for estado, vecinos in grafo.items()}
        for vecino in list(grafo_roto[meta]):
            grafo_roto[vecino].discard(meta)
        grafo_roto[meta] = set()
        problema = ProblemaLaberinto(grafo_roto, inicio, meta)
        for resolver in (busqueda_dfs, busqueda_bfs, busqueda_costo_uniforme,
                         busqueda_bidireccional):
            resultado = resolver(problema)
            self.assertFalse(resultado.encontrado)
            self.assertEqual(resultado.camino, [])
        resultado_lee, etiquetas, _ = lee_frente_de_onda(problema)
        self.assertFalse(resultado_lee.encontrado)
        self.assertNotIn(meta, etiquetas)

    def test_dos_intersecciones_bidireccionales(self):
        resultado = probar_bidir_dos_intersecciones(
            busqueda_bidireccional, ProblemaLaberinto
        )
        self.assertEqual(resultado.profundidad, 2)

    def test_limites_insuficiente_exacto_y_excesivo(self):
        profundidad = busqueda_bfs(self.problema).profundidad
        resultado_corte, hubo_corte = busqueda_limitada_profundidad(
            self.problema, profundidad - 1
        )
        self.assertFalse(resultado_corte.encontrado)
        self.assertTrue(hubo_corte)
        resultado_exacto, _ = busqueda_limitada_profundidad(
            self.problema, profundidad
        )
        resultado_excesivo, _ = busqueda_limitada_profundidad(
            self.problema, profundidad + 5
        )
        self.assertTrue(resultado_exacto.encontrado)
        self.assertTrue(resultado_excesivo.encontrado)
        verificar_camino_valido(self.problema, resultado_exacto)
        verificar_camino_valido(self.problema, resultado_excesivo)


# La ejecucion se realiza al final del punto 9, cuando ya estan definidos sus helpers.
suite = unittest.TestLoader().loadTestsFromTestCase(TestAceptacion)

## 9. Segunda familia de problemas: ciclos y costos

Hunt-and-Kill produce un árbol. Allí existe un único camino entre dos estados y no se aprecia plenamente la optimalidad. Diseñe, sin modificar el generador original, dos transformaciones:

### A. Laberinto con ciclos

Abra aleatoriamente entre 5% y 12% de las paredes internas que no sean corredores. Debe conservar:

- adyacencias ortogonales;
- simetría del grafo;
- reproducibilidad;
- al menos un ciclo comprobable.

### B. Laberinto ponderado

Asigne costos positivos a corredores o terrenos. Construya deliberadamente un caso donde:

```text
camino con menos pasos ≠ camino de menor costo
```

No use pesos negativos. Justifique si los costos pertenecen a celdas, acciones o aristas y mantenga esa decisión en las tres versiones.

In [ ]:
def agregar_ciclos(grafo, filas: int, columnas: int, semilla: int,
                   fraccion: float = 0.05):
    """Abre una fraccion reproducible de paredes interiores, sin mutar grafo."""
    if not 0.05 <= fraccion <= 0.12:
        raise ValueError("La fraccion debe estar entre 0.05 y 0.12")
    grafo_nuevo = {estado: set(vecinos) for estado, vecinos in grafo.items()}
    paredes = []
    for fila in range(filas):
        for columna in range(columnas):
            estado = (fila, columna)
            if columna + 1 < columnas:
                vecino = (fila, columna + 1)
                if vecino not in grafo_nuevo[estado]:
                    paredes.append((estado, vecino))
            if fila + 1 < filas:
                vecino = (fila + 1, columna)
                if vecino not in grafo_nuevo[estado]:
                    paredes.append((estado, vecino))
    if not paredes:
        raise ValueError("No hay paredes cerradas para abrir")
    cantidad_minima = math.ceil(0.05 * len(paredes))
    cantidad_maxima = math.floor(0.12 * len(paredes))
    if cantidad_minima > cantidad_maxima:
        raise ValueError(
            "La malla es demasiado pequena para cumplir el intervalo de 5%-12% "
            "y abrir al menos una pared"
        )
    cantidad = min(
        cantidad_maxima,
        max(cantidad_minima, round(fraccion * len(paredes))),
    )
    rng = random.Random(semilla)
    abiertas = rng.sample(paredes, min(cantidad, len(paredes)))
    for origen, destino in abiertas:
        grafo_nuevo[origen].add(destino)
        grafo_nuevo[destino].add(origen)
    return grafo_nuevo, len(abiertas) / len(paredes), len(abiertas)


def aristas_no_dirigidas(grafo):
    return {
        frozenset((origen, destino))
        for origen, vecinos in grafo.items()
        for destino in vecinos
        if origen != destino
    }


def validar_costos_positivos(grafo, costos) -> None:
    """Rechaza mapas incompletos, no finitos o no estrictamente positivos."""
    requeridas = aristas_no_dirigidas(grafo)
    faltantes = requeridas - set(costos)
    if faltantes:
        raise ValueError(f"Faltan costos para {len(faltantes)} corredores")
    for arista in requeridas:
        costo = costos[arista]
        if not math.isfinite(costo) or costo <= 0:
            raise ValueError(f"Costo invalido para {arista}: {costo}")


def asignar_costos(grafo, semilla: int, minimo: int = 1, maximo: int = 9):
    """Asigna costos enteros positivos reproducibles a las aristas."""
    if minimo <= 0 or maximo < minimo:
        raise ValueError("El rango debe ser positivo y maximo >= minimo")
    rng = random.Random(semilla)
    costos = {
        arista: rng.randint(minimo, maximo)
        for arista in aristas_no_dirigidas(grafo)
    }
    validar_costos_positivos(grafo, costos)
    return costos


def _camino_bfs(problema, arista_bloqueada=None):
    """BFS pequeno para construir rutas de referencia y probar pesos."""
    inicio = problema.inicio
    meta = problema.meta
    cola = deque([inicio])
    padre = {inicio: None}
    while cola:
        actual = cola.popleft()
        if actual == meta:
            camino = []
            while actual is not None:
                camino.append(actual)
                actual = padre[actual]
            return list(reversed(camino))
        for vecino in problema.acciones(actual):
            if arista_bloqueada == frozenset((actual, vecino)):
                continue
            if vecino not in padre:
                padre[vecino] = actual
                cola.append(vecino)
    return []


def forzar_divergencia_pasos_costo(problema, costos_existentes=None):
    """Hace que una ruta de mas pasos cueste menos que toda ruta mas corta.

    Requiere que haya una ruta alternativa estrictamente mas larga entre las
    mismas puntas. Si el grafo no la tiene, se informa en vez de simularla.
    """
    camino_corto = _camino_bfs(problema)
    if not camino_corto:
        raise ValueError("Inicio y meta no estan conectados")
    camino_alternativo = []
    for origen, destino in zip(camino_corto, camino_corto[1:]):
        candidato = _camino_bfs(
            problema, arista_bloqueada=frozenset((origen, destino))
        )
        if len(candidato) > len(camino_corto):
            camino_alternativo = candidato
            break
    if not camino_alternativo:
        raise ValueError(
            "No existe una ruta mas larga alternativa; use un grafo con un ciclo "
            "que conecte dos puntos del recorrido inicio-meta."
        )

    todas = aristas_no_dirigidas(problema.grafo)
    costo_alto = len(problema.grafo) + 1
    costos = {arista: costo_alto for arista in todas}
    for origen, destino in zip(camino_alternativo, camino_alternativo[1:]):
        costos[frozenset((origen, destino))] = 1
    validar_costos_positivos(problema.grafo, costos)
    assert len(camino_alternativo) > len(camino_corto)
    assert sum(costos[frozenset((a, b))] for a, b in zip(
        camino_alternativo, camino_alternativo[1:]
    )) < sum(costos[frozenset((a, b))] for a, b in zip(
        camino_corto, camino_corto[1:]
    ))
    return costos


def problema_demo_ponderado(constructor_problema):
    """Caso controlado: BFS usa 2 pasos; UCS prefiere 3 pasos de costo 1."""
    grafo = {
        "A": {"B", "C"},
        "B": {"A", "D"},
        "C": {"A", "E"},
        "E": {"C", "D"},
        "D": {"B", "E"},
    }
    costos = {
        frozenset(("A", "B")): 10,
        frozenset(("B", "D")): 10,
        frozenset(("A", "C")): 1,
        frozenset(("C", "E")): 1,
        frozenset(("E", "D")): 1,
    }
    validar_costos_positivos(grafo, costos)
    return constructor_problema(grafo, "A", "D", costos=costos)


def validar_laberinto_ciclico(grafo, grafo_arbol, filas: int, columnas: int,
                               fraccion_abierta: float) -> None:
    """Verifica simetria, ortogonalidad, conectividad y al menos un ciclo."""
    assert set(grafo) == set(grafo_arbol)
    for estado, vecinos in grafo.items():
        fila, columna = estado
        assert 0 <= fila < filas and 0 <= columna < columnas
        for vecino in vecinos:
            assert estado in grafo[vecino], f"Arista asimetrica: {estado}->{vecino}"
            assert abs(fila - vecino[0]) + abs(columna - vecino[1]) == 1
    raiz = next(iter(grafo))
    alcanzados = {raiz}
    pila = [raiz]
    while pila:
        actual = pila.pop()
        for vecino in grafo[actual]:
            if vecino not in alcanzados:
                alcanzados.add(vecino)
                pila.append(vecino)
    assert len(alcanzados) == len(grafo), "El grafo debe seguir conexo"
    aristas = sum(len(vecinos) for vecinos in grafo.values()) // 2
    aristas_base = sum(len(vecinos) for vecinos in grafo_arbol.values()) // 2
    assert aristas_base == len(grafo_arbol) - 1, "La base debe ser un arbol"
    assert aristas > aristas_base, "La extension debe contener al menos un ciclo"
    assert 0.05 <= fraccion_abierta <= 0.12


def probar_reproducibilidad_extension(grafo, filas, columnas, semilla):
    grafo_a, fraccion_a, abiertas_a = agregar_ciclos(
        grafo, filas, columnas, semilla, 0.05
    )
    grafo_b, fraccion_b, abiertas_b = agregar_ciclos(
        grafo, filas, columnas, semilla, 0.05
    )
    assert grafo_a == grafo_b and fraccion_a == fraccion_b
    assert abiertas_a == abiertas_b
    costos_a = asignar_costos(grafo_a, semilla)
    costos_b = asignar_costos(grafo_a, semilla)
    assert costos_a == costos_b
    return grafo_a, costos_a


def probar_ucs_entradas_obsoletas(ucs_solver, constructor_problema):
    """Hace que UCS inserte costos mejores repetidos antes de extraer la meta."""
    grafo = {
        "A": {"B", "C", "F"},
        "B": {"A", "D"},
        "C": {"A", "D"},
        "F": {"A", "D"},
        "D": {"B", "C", "F", "G"},
        "G": {"D"},
    }
    costos = {
        frozenset(("A", "B")): 5,
        frozenset(("B", "D")): 5,
        frozenset(("A", "C")): 3,
        frozenset(("C", "D")): 3,
        frozenset(("A", "F")): 1,
        frozenset(("F", "D")): 1,
        frozenset(("D", "G")): 100,
    }
    problema = constructor_problema(grafo, "A", "G", costos=costos)
    resultado = ucs_solver(problema)
    assert resultado.encontrado
    assert resultado.camino == ["A", "F", "D", "G"]
    assert resultado.costo == 102
    verificar_camino_valido(problema, resultado)
    return resultado


def probar_bidir_dos_intersecciones(bidirectional_solver, constructor_problema):
    """Comprueba reconstruccion en un rombo con dos intersecciones posibles."""
    grafo = {
        "A": {"B", "C"},
        "B": {"A", "D"},
        "C": {"A", "D"},
        "D": {"B", "C"},
    }
    problema = constructor_problema(grafo, "A", "D")
    resultado = bidirectional_solver(problema)
    assert resultado.encontrado
    assert resultado.camino in (["A", "B", "D"], ["A", "C", "D"])
    verificar_camino_valido(problema, resultado)
    return resultado


def probar_limites_dls(problema, bfs_solver, dls_solver):
    """Comprueba limite insuficiente, exacto y excesivo en DLS propio."""
    profundidad = bfs_solver(problema).profundidad
    resultado_corte, hubo_corte = dls_solver(problema, profundidad - 1)
    assert not resultado_corte.encontrado and hubo_corte
    resultado_exacto, _ = dls_solver(problema, profundidad)
    assert resultado_exacto.encontrado
    resultado_excesivo, _ = dls_solver(problema, profundidad + 5)
    assert resultado_excesivo.encontrado
    for resultado in (resultado_exacto, resultado_excesivo):
        verificar_camino_valido(problema, resultado)
    return resultado_corte, resultado_exacto, resultado_excesivo


def probar_divergencia_bfs_ucs(constructor_problema, bfs_solver, ucs_solver):
    problema = problema_demo_ponderado(constructor_problema)
    resultado_bfs = bfs_solver(problema)
    resultado_ucs = ucs_solver(problema)
    assert resultado_bfs.profundidad == 2
    assert resultado_ucs.profundidad == 3
    assert resultado_ucs.costo < resultado_bfs.costo
    verificar_camino_valido(problema, resultado_bfs)
    verificar_camino_valido(problema, resultado_ucs)
    return resultado_bfs, resultado_ucs

In [ ]:
# Punto 9: validaciones de extension, pruebas del punto 8 e integracion opcional.

def probar_ucs_entradas_obsoletas(ucs_solver, constructor_problema):
    """Fuerza dos entradas obsoletas para D antes de extraer la meta G."""
    grafo_prueba = {
        "A": {"B", "C", "F"}, "B": {"A", "D"},
        "C": {"A", "D"}, "F": {"A", "D"},
        "D": {"B", "C", "F", "G"}, "G": {"D"},
    }
    costos_prueba = {
        frozenset(("A", "B")): 1, frozenset(("B", "D")): 9,
        frozenset(("A", "C")): 2, frozenset(("C", "D")): 3,
        frozenset(("A", "F")): 3, frozenset(("F", "D")): 1,
        frozenset(("D", "G")): 100,
    }
    problema_prueba = constructor_problema(
        grafo_prueba, "A", "G", costos=costos_prueba
    )
    resultado = ucs_solver(problema_prueba)
    assert resultado.encontrado
    assert resultado.camino == ["A", "F", "D", "G"]
    assert resultado.costo == 104
    assert resultado.repetidos_descartados >= 2
    verificar_camino_valido(problema_prueba, resultado)
    return resultado


grafo_ciclico, fraccion_abierta, paredes_abiertas = agregar_ciclos(
    grafo, filas, columnas, semilla, fraccion=0.05
)
validar_laberinto_ciclico(grafo_ciclico, grafo, filas, columnas, fraccion_abierta)
grafo_reproducible, costos_reproducibles = probar_reproducibilidad_extension(
    grafo, filas, columnas, semilla
)
assert grafo_reproducible == agregar_ciclos(
    grafo, filas, columnas, semilla, fraccion=0.05
)[0]
validar_costos_positivos(grafo_reproducible, costos_reproducibles)

resultado_bfs_ponderado, resultado_ucs_ponderado = probar_divergencia_bfs_ucs(
    ProblemaLaberinto, busqueda_bfs, busqueda_costo_uniforme
)
resultado_ucs_obsoleto = probar_ucs_entradas_obsoletas(
    busqueda_costo_uniforme, ProblemaLaberinto
)
print(
    f"Punto 9 OK: {paredes_abiertas} paredes abiertas ({fraccion_abierta:.1%}); "
    f"BFS={resultado_bfs_ponderado.profundidad} pasos/costo "
    f"{resultado_bfs_ponderado.costo:g}; UCS={resultado_ucs_ponderado.profundidad} "
    f"pasos/costo {resultado_ucs_ponderado.costo:g}; "
    f"obsoletos UCS={resultado_ucs_obsoleto.repetidos_descartados}"
)

suite = unittest.TestLoader().loadTestsFromTestCase(TestAceptacion)
resultado_suite = unittest.TextTestRunner(verbosity=2).run(suite)
assert resultado_suite.wasSuccessful(), "Fallaron pruebas del punto 8"

# Si las bibliotecas externas estan instaladas, validar tambien caminos y costos.
def validar_resultados_biblioteca(nombre, resultados):
    for algoritmo, resultado in resultados.items():
        with unittest.TestCase().subTest(version=nombre, algoritmo=algoritmo):
            assert resultado.encontrado, f"{nombre}/{algoritmo} no encontro la meta"
            verificar_camino_valido(problema_lee, resultado)
    if "BFS" in resultados:
        assert resultados["BFS"].profundidad == resultado_bfs_verificacion.profundidad

    problema_ponderado = problema_demo_ponderado(ProblemaLaberinto)
    bfs_ponderado = (ejecutar_simpleai(problema_ponderado, "BFS")
                     if nombre == "SimpleAI"
                     else ejecutar_aima(problema_ponderado, "BFS"))
    ucs_ponderado = (ejecutar_simpleai(problema_ponderado, "UCS")
                     if nombre == "SimpleAI"
                     else ejecutar_aima(problema_ponderado, "UCS"))
    verificar_camino_valido(problema_ponderado, bfs_ponderado)
    verificar_camino_valido(problema_ponderado, ucs_ponderado)
    assert ucs_ponderado.costo <= bfs_ponderado.costo


if resultados_simpleai:
    validar_resultados_biblioteca("SimpleAI", resultados_simpleai)
    print("SimpleAI: caminos unitarios y ponderados verificados.")
else:
    print("SimpleAI sin validar: falta instalar simpleai y reejecutar su celda.")
if resultados_aima:
    validar_resultados_biblioteca("AIMA-Python", resultados_aima)
    print("AIMA-Python: caminos unitarios y ponderados verificados.")
else:
    print("AIMA-Python sin validar: falta instalar el repositorio y reejecutar su celda.")

## 10. Diseño experimental

Ejecute como mínimo **30 instancias** por configuración y reporte mediana y rango intercuartílico, no solamente un caso.

| Factor | Niveles mínimos |
|---|---|
| Tamaño | 15×15, 25×25, 40×40 |
| Topología | árbol, 5% ciclos, 10% ciclos |
| Costos | unitarios, ponderados |
| Posición | esquinas, interior, cercanas |
| Algoritmos | todos los exigidos que sean aplicables |

Debe controlar semillas y separar tiempo de generación del tiempo de búsqueda.

### Gráficas obligatorias

1. expandidos frente a número de estados;
2. frontera máxima frente a profundidad de solución;
3. tiempo frente a tamaño;
4. costo y longitud de las soluciones;
5. mapa de calor de Lee;
6. comparación cruzada de las tres versiones.

### Discusión obligatoria

- ¿Qué métrica resulta más estable que el tiempo?
- ¿Cuándo la búsqueda bidireccional pierde su ventaja?
- ¿Por qué IDDFS repite trabajo y aun así puede ser conveniente?
- ¿Qué efecto tiene el orden de sucesores sobre DFS?
- ¿Qué parte de las diferencias se debe al algoritmo y cuál a la biblioteca?

In [ ]:
# Punto 10: experimento reproducible, resumen mediana/IQR, CSV y seis graficas.
# La celda ejecuta una prueba rapida para mostrar las figuras; el protocolo
# completo de 30 semillas esta en ejecutar_punto10_completo().
import csv
import math
import statistics
import time
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display


def elegir_extremos_experimento(grafo, filas, columnas, posicion, semilla):
    rng = random.Random(semilla + 17)
    if posicion == "esquinas":
        return (0, 0), (filas - 1, columnas - 1)
    if posicion == "interior":
        mitad_fila, mitad_columna = filas // 2, columnas // 2
        arriba_izq = [s for s in grafo if s[0] < mitad_fila and s[1] < mitad_columna]
        abajo_der = [s for s in grafo if s[0] >= mitad_fila and s[1] >= mitad_columna]
        return rng.choice(arriba_izq), rng.choice(abajo_der)
    if posicion == "cercanas":
        inicio_local = rng.choice(list(grafo))
        distancias, cola = {inicio_local: 0}, deque([inicio_local])
        while cola:
            actual = cola.popleft()
            for vecino in sorted(grafo[actual]):
                if vecino not in distancias:
                    distancias[vecino] = distancias[actual] + 1
                    cola.append(vecino)
        candidatas = [s for s, d in distancias.items() if 3 <= d <= 6]
        if not candidatas:
            candidatas = [s for s in distancias if s != inicio_local]
        return inicio_local, rng.choice(candidatas)
    raise ValueError(f"Posicion desconocida: {posicion}")


def ejecutar_matriz_experimental(
    solucionadores, semillas=range(30), tamanos=(15, 25, 40),
    topologias=(("arbol", 0.0), ("ciclos_5pct", 0.05), ("ciclos_10pct", 0.10)),
    modos_costo=("unitario", "ponderado"),
    posiciones=("esquinas", "interior", "cercanas"),
):
    """Ejecuta las configuraciones emparejadas y devuelve una fila por corrida."""
    filas_resultado = []
    for tamano in tamanos:
        for semilla_local in tuple(semillas):
            reloj = time.perf_counter()
            grafo_base = LaberintoHuntKill(tamano, tamano, semilla_local).generar()
            tiempo_generador_ms = (time.perf_counter() - reloj) * 1000
            for nombre_topologia, fraccion in topologias:
                reloj = time.perf_counter()
                if fraccion:
                    grafo_local, fraccion_real, abiertas = agregar_ciclos(
                        grafo_base, tamano, tamano, semilla_local, fraccion
                    )
                else:
                    grafo_local = {s: set(v) for s, v in grafo_base.items()}
                    fraccion_real, abiertas = 0.0, 0
                tiempo_topologia_ms = (time.perf_counter() - reloj) * 1000
                for modo in modos_costo:
                    reloj = time.perf_counter()
                    costos = None if modo == "unitario" else asignar_costos(
                        grafo_local, semilla_local
                    )
                    tiempo_costos_ms = (time.perf_counter() - reloj) * 1000
                    for posicion in posiciones:
                        reloj = time.perf_counter()
                        inicio_local, meta_local = elegir_extremos_experimento(
                            grafo_local, tamano, tamano, posicion, semilla_local
                        )
                        problema_local = ProblemaLaberinto(
                            grafo_local, inicio_local, meta_local, costos=costos
                        )
                        tiempo_posiciones_ms = (time.perf_counter() - reloj) * 1000
                        tiempo_preparacion_ms = (
                            tiempo_generador_ms + tiempo_topologia_ms
                            + tiempo_costos_ms + tiempo_posiciones_ms
                        )
                        for version, algoritmo, resolver in solucionadores:
                            if modo == "ponderado" and algoritmo in ("Lee", "BIDIRECCIONAL"):
                                continue
                            reloj = time.perf_counter()
                            resultado = resolver(problema_local)
                            tiempo_medido_ms = (time.perf_counter() - reloj) * 1000
                            verificar_camino_valido(problema_local, resultado)
                            filas_resultado.append({
                                "semilla": semilla_local, "tamano": tamano,
                                "estados": len(grafo_local), "topologia": nombre_topologia,
                                "fraccion_paredes_abiertas": fraccion_real,
                                "paredes_abiertas": abiertas, "modo_costo": modo,
                                "posicion": posicion, "inicio": repr(inicio_local),
                                "meta": repr(meta_local), "version": version,
                                "algoritmo": algoritmo, "encontrado": resultado.encontrado,
                                "costo": resultado.costo, "profundidad": resultado.profundidad,
                                "expandidos": resultado.expandidos,
                                "generados": resultado.generados,
                                "repetidos_descartados": resultado.repetidos_descartados,
                                "frontera_maxima": resultado.frontera_maxima,
                                "tiempo_busqueda_ms": resultado.tiempo_ms or tiempo_medido_ms,
                                "tiempo_preparacion_ms": tiempo_preparacion_ms,
                            })
    return filas_resultado


def percentil_lineal(valores, proporcion):
    validos = sorted(
        valor for valor in valores
        if valor is not None and math.isfinite(valor)
    )
    if not validos:
        return None
    posicion = (len(validos) - 1) * proporcion
    inferior, superior = math.floor(posicion), math.ceil(posicion)
    if inferior == superior:
        return validos[inferior]
    peso = posicion - inferior
    return validos[inferior] * (1 - peso) + validos[superior] * peso


def resumir_matriz_experimental(filas_resultado):
    claves = ("tamano", "topologia", "modo_costo", "posicion", "version", "algoritmo")
    medidas = ("expandidos", "generados", "repetidos_descartados", "frontera_maxima",
               "tiempo_busqueda_ms", "costo", "profundidad")
    grupos = {}
    for fila in filas_resultado:
        grupos.setdefault(tuple(fila[k] for k in claves), []).append(fila)
    resumen = []
    for clave, registros in grupos.items():
        salida = dict(zip(claves, clave))
        salida["n"] = len(registros)
        for medida in medidas:
            valores = [r[medida] for r in registros]
            validos = [v for v in valores if v is not None]
            q1, q3 = percentil_lineal(valores, 0.25), percentil_lineal(valores, 0.75)
            salida[f"{medida}_mediana"] = statistics.median(validos) if validos else None
            salida[f"{medida}_iqr"] = q3 - q1 if q1 is not None and q3 is not None else None
        resumen.append(salida)
    return resumen


def validar_cobertura_experimental(filas_resultado, semillas_esperadas=30):
    grupos = {}
    for fila in filas_resultado:
        clave = (fila["tamano"], fila["topologia"], fila["modo_costo"], fila["posicion"])
        grupos.setdefault(clave, set()).add(fila["semilla"])
    incompletos = {clave: len(semillas) for clave, semillas in grupos.items()
                   if len(semillas) < semillas_esperadas}
    assert not incompletos, f"Faltan semillas en configuraciones: {incompletos}"
    return len(grupos)


def guardar_csv_punto10(filas_resultado, resumen, sufijo=""):
    if not filas_resultado or not resumen:
        raise ValueError("No hay datos para exportar")
    with open(f"resultados_punto10{sufijo}.csv", "w", newline="", encoding="utf-8") as archivo:
        escritor = csv.DictWriter(archivo, fieldnames=list(filas_resultado[0]))
        escritor.writeheader()
        escritor.writerows(filas_resultado)
    with open(f"resumen_punto10{sufijo}.csv", "w", newline="", encoding="utf-8") as archivo:
        escritor = csv.DictWriter(archivo, fieldnames=list(resumen[0]))
        escritor.writeheader()
        escritor.writerows(resumen)


def graficar_resultados_punto10(filas_resultado, etiquetas_lee, filas, columnas):
    """Genera: expansiones, frontera, tiempo, costo/pasos, Lee y versiones."""
    fig, ejes = plt.subplots(2, 3, figsize=(18, 10), layout="constrained")
    ejes = list(ejes.flat)
    grupos = sorted({(r["version"], r["algoritmo"]) for r in filas_resultado})
    for version, algoritmo in grupos:
        datos = [r for r in filas_resultado if r["version"] == version
                 and r["algoritmo"] == algoritmo and r["modo_costo"] == "unitario"]
        ejes[0].scatter([r["estados"] for r in datos], [r["expandidos"] for r in datos],
                        s=10, alpha=0.35, label=f"{version}/{algoritmo}")
        frontera = [r for r in datos if r["frontera_maxima"] is not None]
        if frontera:
            ejes[1].scatter([r["profundidad"] for r in frontera],
                            [r["frontera_maxima"] for r in frontera],
                            s=10, alpha=0.35, label=f"{version}/{algoritmo}")
        tamanos_plot, medianas = [], []
        for tamano in sorted({r["tamano"] for r in filas_resultado}):
            tiempos = [r["tiempo_busqueda_ms"] for r in filas_resultado
                       if r["version"] == version and r["algoritmo"] == algoritmo
                       and r["tamano"] == tamano and r["topologia"] == "arbol"
                       and r["modo_costo"] == "unitario" and r["posicion"] == "esquinas"]
            if tiempos:
                tamanos_plot.append(tamano)
                medianas.append(statistics.median(tiempos))
        if tamanos_plot:
            ejes[2].plot(tamanos_plot, medianas, marker="o", label=f"{version}/{algoritmo}")
    ejes[0].set(title="Expandidos vs estados", xlabel="Estados", ylabel="Expandidos")
    ejes[1].set(title="Frontera maxima vs profundidad", xlabel="Profundidad", ylabel="Frontera")
    ejes[2].set(title="Tiempo vs tamano (mediana)", xlabel="Lado del laberinto", ylabel="ms")

    for version in sorted({r["version"] for r in filas_resultado}):
        datos = [r for r in filas_resultado if r["version"] == version
                 and r["modo_costo"] == "ponderado" and r["encontrado"]]
        if datos:
            ejes[3].scatter([r["profundidad"] for r in datos], [r["costo"] for r in datos],
                            s=12, alpha=0.4, label=version)
    ejes[3].set(title="Costo vs pasos (ponderado)", xlabel="Pasos", ylabel="Costo")
    mapa = np.full((filas, columnas), np.nan)
    for (fila, columna), etiqueta in etiquetas_lee.items():
        mapa[fila, columna] = etiqueta
    imagen = ejes[4].imshow(mapa, interpolation="nearest", cmap="plasma")
    fig.colorbar(imagen, ax=ejes[4], label="Distancia desde inicio")
    ejes[4].set(title="Lee: tiempo de llegada", xlabel="Columna", ylabel="Fila")

    versiones = sorted({r["version"] for r in filas_resultado
                        if r["algoritmo"] == "BFS" and r["tamano"] == 25
                        and r["topologia"] == "arbol" and r["modo_costo"] == "unitario"
                        and r["posicion"] == "esquinas" and r["expandidos"] is not None})
    medianas_bfs = []
    for version in versiones:
        datos = [r["expandidos"] for r in filas_resultado
                 if r["version"] == version and r["algoritmo"] == "BFS"
                 and r["tamano"] == 25 and r["topologia"] == "arbol"
                 and r["modo_costo"] == "unitario" and r["posicion"] == "esquinas"
                 and r["expandidos"] is not None]
        medianas_bfs.append(statistics.median(datos))
    if versiones:
        ejes[5].bar(versiones, medianas_bfs)
    else:
        ejes[5].text(0.5, 0.5, "Sin resultados BFS", ha="center", va="center")
    ejes[5].set(title="Comparacion BFS entre versiones", ylabel="Mediana expandidos")
    for ax in ejes:
        ax.tick_params(axis="x", labelrotation=25)
    fig.suptitle("Experimentos de busqueda no informada")
    plt.show()


def crear_solucionadores_experimento():
    solucionadores = [
        ("Desde cero", "DFS", busqueda_dfs),
        ("Desde cero", "BFS", busqueda_bfs),
        ("Desde cero", "UCS", busqueda_costo_uniforme),
        ("Desde cero", "DLS", lambda p: busqueda_limitada_profundidad(p, len(p.grafo))[0]),
        ("Desde cero", "IDDFS", busqueda_profundizacion_iterativa),
        ("Desde cero", "BIDIRECCIONAL", busqueda_bidireccional),
        ("Desde cero", "Lee", lambda p: lee_frente_de_onda(p)[0]),
    ]
    # Solo se registran los algoritmos que la biblioteca realmente ejecuto bien
    # en la Version 2/3 (p. ej. aima3 no soporta BIDIRECCIONAL y fue omitido).
    for nombre in ("DFS", "BFS", "UCS", "DLS", "IDDFS"):
        if nombre in resultados_simpleai:
            solucionadores.append(("SimpleAI", nombre,
                lambda p, algoritmo=nombre: ejecutar_simpleai(p, algoritmo)))
    for nombre in ("DFS", "BFS", "UCS", "DLS", "IDDFS", "BIDIRECCIONAL"):
        if nombre in resultados_aima:
            solucionadores.append(("AIMA-Python", nombre,
                lambda p, algoritmo=nombre: ejecutar_aima(p, algoritmo)))
    return solucionadores


def ejecutar_punto10_completo():
    """Corre el diseno completo: 30 semillas por las 54 configuraciones."""
    solucionadores = crear_solucionadores_experimento()
    resultados = []
    for tamano in (15, 25, 40):
        lote = ejecutar_matriz_experimental(
            solucionadores, semillas=range(30), tamanos=(tamano,)
        )
        resultados.extend(lote)
        resumen_lote = resumir_matriz_experimental(resultados)
        guardar_csv_punto10(resultados, resumen_lote, sufijo="_completo")
        print(f"Tamano {tamano}x{tamano} terminado: {len(resultados)} filas guardadas")
    configuraciones = validar_cobertura_experimental(resultados, 30)
    assert configuraciones == 54
    return resultados, resumir_matriz_experimental(resultados)


solucionadores_experimento = crear_solucionadores_experimento()
print("Solvers disponibles:", ", ".join(
    f"{version}/{algoritmo}" for version, algoritmo, _ in solucionadores_experimento
))
# Prueba de graficas rapida; el protocolo evaluable completo es ejecutar_punto10_completo().
resultados_piloto = ejecutar_matriz_experimental(
    solucionadores_experimento,
    semillas=range(2), tamanos=(15,),
)
configuraciones_piloto = validar_cobertura_experimental(resultados_piloto, 2)
resumen_piloto = resumir_matriz_experimental(resultados_piloto)
guardar_csv_punto10(resultados_piloto, resumen_piloto, sufijo="_piloto")
print(
    f"Piloto grafico: {len(resultados_piloto)} mediciones en "
    f"{configuraciones_piloto} configuraciones y 2 semillas."
)
comparacion_piloto = [r for r in resumen_piloto if r["topologia"] == "arbol"
                      and r["modo_costo"] == "unitario" and r["posicion"] == "esquinas"]
try:
    import pandas as pd
    display(pd.DataFrame(comparacion_piloto))
except ImportError:
    for registro in comparacion_piloto:
        print(registro)
graficar_resultados_punto10(resultados_piloto, etiquetas_lee, filas, columnas)
print(
    "La tabla piloto sirve para comprobar las graficas; para el minimo evaluable "
    "de 30 semillas, ejecuta ejecutar_punto10_completo()."
)

In [ ]:
# Ejecucion completa por lotes con reanudacion por CSV tras reiniciar el kernel.
import os

resultados_punto10_completo = globals().get("resultados_punto10_completo", [])
tamanos_punto10_completados = globals().get("tamanos_punto10_completados", set())


def _cargar_checkpoint_punto10(tamano):
    rutas = (
        f"resultados_punto10_tamano{tamano}_checkpoint.csv",
        f"resultados_punto10_tamano{tamano}.csv",
    )
    campos_enteros = {
        "semilla", "tamano", "estados", "paredes_abiertas", "profundidad",
        "expandidos", "generados", "repetidos_descartados", "frontera_maxima",
    }
    campos_reales = {
        "fraccion_paredes_abiertas", "costo", "tiempo_busqueda_ms",
        "tiempo_preparacion_ms",
    }
    for ruta in rutas:
        if not os.path.exists(ruta):
            continue
        with open(ruta, newline="", encoding="utf-8") as archivo:
            filas_csv = list(csv.DictReader(archivo))
        for fila in filas_csv:
            for campo in campos_enteros | campos_reales:
                valor = fila.get(campo)
                if valor in (None, "", "None"):
                    fila[campo] = None
                elif campo in campos_enteros:
                    fila[campo] = int(valor)
                else:
                    fila[campo] = float(valor)
            fila["encontrado"] = fila.get("encontrado", "").lower() == "true"
        if filas_csv:
            print(f"Checkpoint recuperado: {ruta} ({len(filas_csv)} mediciones)")
            return filas_csv
    return []


def ejecutar_lote_punto10(tamano):
    """Ejecuta/reanuda 30 semillas para las 18 configuraciones de un tamaño."""
    if tamano not in (15, 25, 40):
        raise ValueError("Los tamanos validos son 15, 25 y 40")
    resultados_lote = [
        fila for fila in resultados_punto10_completo if fila["tamano"] == tamano
    ]
    if not resultados_lote:
        resultados_lote = _cargar_checkpoint_punto10(tamano)
        resultados_punto10_completo.extend(resultados_lote)
    semillas_hechas = {fila["semilla"] for fila in resultados_lote}
    solucionadores_lote = crear_solucionadores_experimento()

    for semilla_lote in range(30):
        if semilla_lote in semillas_hechas:
            continue
        filas_semilla = ejecutar_matriz_experimental(
            solucionadores_lote,
            semillas=(semilla_lote,),
            tamanos=(tamano,),
        )
        resultados_punto10_completo.extend(filas_semilla)
        resultados_lote.extend(filas_semilla)
        semillas_hechas.add(semilla_lote)
        resumen_lote = resumir_matriz_experimental(resultados_lote)
        guardar_csv_punto10(
            resultados_lote, resumen_lote, sufijo=f"_tamano{tamano}_checkpoint"
        )
        if len(semillas_hechas) % 5 == 0:
            print(
                f"{tamano}x{tamano}: completadas {len(semillas_hechas)}/30 semillas "
                f"({len(resultados_lote)} mediciones guardadas)"
            )

    configuraciones = validar_cobertura_experimental(resultados_lote, 30)
    assert configuraciones == 18
    tamanos_punto10_completados.add(tamano)
    resumen_lote = resumir_matriz_experimental(resultados_lote)
    guardar_csv_punto10(resultados_lote, resumen_lote, sufijo=f"_tamano{tamano}")
    print(f"Lote {tamano}x{tamano} listo: 18 configuraciones y 30 semillas.")
    return resumen_lote


def finalizar_punto10():
    """Verifica 54 configuraciones, exporta la matriz completa y grafica."""
    faltantes = {15, 25, 40} - tamanos_punto10_completados
    if faltantes:
        raise RuntimeError(f"Faltan lotes por ejecutar: {sorted(faltantes)}")
    configuraciones = validar_cobertura_experimental(resultados_punto10_completo, 30)
    assert configuraciones == 54
    resumen = resumir_matriz_experimental(resultados_punto10_completo)
    guardar_csv_punto10(resultados_punto10_completo, resumen, sufijo="_completo")
    graficar_resultados_punto10(
        resultados_punto10_completo, etiquetas_lee, filas, columnas
    )
    central = [
        fila for fila in resumen
        if fila["tamano"] == 25 and fila["topologia"] == "arbol"
        and fila["modo_costo"] == "unitario" and fila["posicion"] == "esquinas"
    ]
    try:
        import pandas as pd
        display(pd.DataFrame(central))
    except ImportError:
        for fila in central:
            print(fila)
    print(
        f"Punto 10 completo: {len(resultados_punto10_completo)} mediciones, "
        "54 configuraciones y 30 semillas por configuracion."
    )
    return resultados_punto10_completo, resumen


def ejecutar_punto10_completo():
    """Reanuda automaticamente los tamaños pendientes desde sus CSV checkpoint."""
    for tamano in (15, 25, 40):
        if tamano not in tamanos_punto10_completados:
            ejecutar_lote_punto10(tamano)
    return finalizar_punto10()

In [ ]:
# Lote 15x15 ya ejecutado y guardado en CSV.
# Para recalcularlo manualmente, ejecutar: ejecutar_lote_punto10(15)

In [ ]:
# Lote 25x25 ya ejecutado y guardado en CSV.
# Para recalcularlo manualmente, ejecutar: ejecutar_lote_punto10(25)

In [ ]:
# Lote 40x40 ya ejecutado y guardado en CSV.
# Para recalcularlo manualmente, ejecutar: ejecutar_lote_punto10(40)

## Interpretación del punto 10

El diseño completo cruza 3 tamaños, 3 topologías, 2 esquemas de costo y 3 posiciones: **54 configuraciones**. Con 30 semillas emparejadas son **1620 instancias**. Para cada una se ejecutan los algoritmos aplicables; el total de este cuaderno fue 9720 mediciones. El código separa preparación/generación de búsqueda, valida cada camino y calcula mediana e IQR.

La figura de la prueba rápida con dos semillas solo comprueba el renderizado; no es evidencia estadística. El estudio completo está en `resultados_punto10_completo.csv` (9720 mediciones crudas) y `resumen_punto10_completo.csv` (324 grupos con mediana e IQR).

Los archivos `*_piloto.csv` son de prueba; `*_tamano15.csv`, `*_tamano25.csv`, `*_tamano40.csv` y `*_checkpoint.csv` son lotes intermedios/de recuperación. Tras verificar los dos CSV completos, puedes eliminar esos auxiliares; conserva el crudo y el resumen para respaldar la entrega.

Para una demostración con otra semilla, ejecuta `ejecutar_demostracion_semilla(12345)`. Corre los algoritmos y compara las tres versiones en aproximadamente un segundo sin volver a ejecutar el punto 10. Para incluir las figuras usa `ejecutar_demostracion_semilla(12345, graficar=True)`; el trazado puede tardar más que las búsquedas.

Las celdas de lotes 15×15, 25×25 y 40×40 están desactivadas para que “Run All” no repita el experimento largo. Para reanudar manualmente, ejecutar `ejecutar_lote_punto10(15)`, `ejecutar_lote_punto10(25)` y `ejecutar_lote_punto10(40)`; el helper recupera semillas ya guardadas en CSV. Al terminar, `finalizar_punto10()` valida las 54 configuraciones y muestra las seis gráficas.

### Discusión

- **Métrica más estable:** el número de expandidos suele variar menos con carga del sistema y hardware que el tiempo. La frontera máxima ayuda a comparar memoria; en adaptadores externos se marca como no disponible si la API no la expone.
- **Cuándo pierde ventaja bidireccional:** cuando el espacio no es reversible, no se puede buscar desde una meta concreta, las ramas tienen tamaños muy desiguales, la intersección es costosa de detectar o el problema es tan pequeño que gestionar dos fronteras añade sobrecarga. Esta implementación solo garantiza optimalidad por pasos con costos unitarios.
- **Por qué IDDFS repite trabajo:** cada límite vuelve a expandir niveles anteriores. A cambio, conserva espacio aproximadamente lineal en la profundidad y es completo/óptimo por pasos con factor de ramificación finito y costos unitarios.
- **Efecto del orden de sucesores en DFS:** determina qué ramas se exploran primero y puede cambiar tanto la solución encontrada como el número de expansiones. No cambia que cada arista del camino deba ser válida; DFS no promete el camino más corto.
- **Algoritmo frente a biblioteca:** la disciplina de frontera y repetidos pertenecen al algoritmo. Desempates, sobrecarga, representación de nodos, orden de sucesores y métricas disponibles dependen de la biblioteca e intérprete. Por ello se mantienen fijos grafo, semilla, inicio, meta, costos y orden de acciones; las llamadas a `actions` se reportan como llamadas, no como nodos aceptados.

In [ ]:
# Ejecucion rapida para demostracion oral; las figuras son opcionales.
def ejecutar_demostracion_semilla(semilla_nueva, graficar=False):
    """Reconstruye una instancia y compara algoritmos sin correr el punto 10."""
    digitos = str(semilla_nueva).zfill(5)
    filas_demo = 20 + int(digitos[-1])
    columnas_demo = 20 + int(digitos[-2])
    grafo_demo = LaberintoHuntKill(
        filas=filas_demo, columnas=columnas_demo, semilla=semilla_nueva
    ).generar()
    rng_demo = random.Random(semilla_nueva + 1)
    cuadrante_inicio_demo = rng_demo.choice((1, 2, 3, 4))
    cuadrantes_opuestos = {1: 4, 2: 3, 3: 2, 4: 1}
    cuadrante_meta_demo = cuadrantes_opuestos[cuadrante_inicio_demo]
    inicios = [
        estado for estado in grafo_demo
        if cuadrante_de(estado, filas_demo, columnas_demo) == cuadrante_inicio_demo
    ]
    metas = [
        estado for estado in grafo_demo
        if cuadrante_de(estado, filas_demo, columnas_demo) == cuadrante_meta_demo
    ]
    inicio_demo = rng_demo.choice(inicios)
    meta_demo = rng_demo.choice(metas)
    problema_demo = ProblemaLaberinto(grafo_demo, inicio_demo, meta_demo)
    resolvers = {
        "DFS": busqueda_dfs,
        "BFS": busqueda_bfs,
        "UCS": busqueda_costo_uniforme,
        "DLS": lambda p: busqueda_limitada_profundidad(p, len(p.grafo))[0],
        "IDDFS": busqueda_profundizacion_iterativa,
        "BIDIRECCIONAL": busqueda_bidireccional,
        "Lee": lambda p: lee_frente_de_onda(p)[0],
    }
    resultados_demo = {}
    for nombre, resolver in resolvers.items():
        resultado = resolver(problema_demo)
        verificar_camino_valido(problema_demo, resultado)
        resultados_demo[nombre] = resultado
        print(
            f"Desde cero/{nombre}: pasos={resultado.profundidad}, "
            f"costo={resultado.costo:g}, expandidos={resultado.expandidos}, "
            f"tiempo_ms={resultado.tiempo_ms:.3f}"
        )

    resultado_lee_demo, etiquetas_demo, historial_demo = lee_frente_de_onda(
        problema_demo
    )
    assert resultado_lee_demo.profundidad == resultados_demo["BFS"].profundidad

    if resultados_simpleai:
        for nombre in ("DFS", "BFS", "UCS", "DLS", "IDDFS"):
            resultado = ejecutar_simpleai(problema_demo, nombre)
            verificar_camino_valido(problema_demo, resultado)
            resultados_demo[f"SimpleAI/{nombre}"] = resultado
            print(
                f"SimpleAI/{nombre}: pasos={resultado.profundidad}, "
                f"costo={resultado.costo:g}, llamadas_actions="
                f"{resultado.llamadas_acciones}, tiempo_ms={resultado.tiempo_ms:.3f}"
            )
    if resultados_aima:
        for nombre in ("DFS", "BFS", "UCS", "DLS", "IDDFS"):
            resultado = ejecutar_aima(problema_demo, nombre)
            verificar_camino_valido(problema_demo, resultado)
            resultados_demo[f"AIMA-Python/{nombre}"] = resultado
            print(
                f"AIMA-Python/{nombre}: pasos={resultado.profundidad}, "
                f"costo={resultado.costo:g}, llamadas_actions="
                f"{resultado.llamadas_acciones}, tiempo_ms={resultado.tiempo_ms:.3f}"
            )
    if graficar:
        graficar_frentes_lee(
            historial_demo, grafo_demo, filas_demo, columnas_demo, inicio_demo,
            meta=meta_demo, maximo_instantes=8,
        )
        graficar_laberinto_con_camino(
            grafo_demo, filas_demo, columnas_demo, inicio_demo, meta_demo,
            resultado_lee_demo.camino,
        )
        versiones = {
            nombre: resultado for nombre, resultado in resultados_demo.items()
            if "/" in nombre
        }
        if versiones:
            graficar_caminos_versiones(
                grafo_demo, filas_demo, columnas_demo, inicio_demo, meta_demo,
                versiones, f"Rutas comparadas, semilla {semilla_nueva}",
            )
    return problema_demo, resultados_demo, etiquetas_demo


print(
    "Demostracion rapida: ejecutar_demostracion_semilla(12345). "
    "Para graficas, usar graficar=True."
)

In [ ]:
semilla_demostracion = 12345
problema_demostracion, resultados_demostracion, etiquetas_demostracion = ejecutar_demostracion_semilla(semilla_demostracion)

## 11. Análisis teórico

Usamos $b$ como factor de ramificación, $d$ como profundidad de la solución menos profunda, $m$ como profundidad máxima, $\ell$ como límite de DLS, $C^*$ como costo óptimo y $\varepsilon>0$ como costo mínimo por acción. Las cotas de DFS/BFS/DLS/IDDFS se dan para búsqueda en árbol; en un grafo finito con repetidos, recorrer el grafo explícito cuesta $O(|V|+|E|)$ tiempo y hasta $O(|V|)$ memoria.

| Algoritmo | Completo | Óptimo | Tiempo | Espacio | Condiciones |
|---|---|---|---|---|---|
| DFS | No en profundidad infinita; sí en grafo finito con control de repetidos | No | $O(b^m)$ | $O(bm)$ en árbol; $O(|V|)$ con alcanzados | Ramificación finita; controlar ciclos |
| BFS | Sí, si $b$ es finito | Sí, si todos los pasos cuestan lo mismo | $O(b^{d+1})$ | $O(b^{d+1})$ | Encuentra la ruta con menor profundidad; memoria alta |
| UCS | Sí si cada costo es al menos $\varepsilon>0$ | Sí con costos no negativos, al extraer la meta | $O(b^{1+\lfloor C^*/\varepsilon\rfloor})$ | Misma cota que los nodos guardados | No detenerse al generar la meta; conservar las mejoras de costo |
| DLS | Sí si $\ell\ge d$ | No | $O(b^\ell)$ | $O(b\ell)$ | Distinguir corte de fracaso; no prometer solución si $\ell<d$ |
| IDDFS | Sí si $b$ es finito | Sí con costo unitario | $O(b^d)$ | $O(bd)$ | Repite niveles poco profundos; memoria menor que BFS |
| Bidireccional | Sí si el grafo es reversible y ambos extremos son conocidos | Sí por pasos con BFS y costos unitarios | $O(b^{d/2})$ | $O(b^{d/2})$ | Detectar intersección; no es UCS para pesos distintos |
| Lee | Sí en grafo finito si la meta es alcanzable | Sí por número de aristas con costos unitarios | $O(|V|+|E|)$ | $O(|V|)$ más historial visual | Etiqueta frentes por distancia; no minimiza costo ponderado |

**Relación con el experimento.** En el subconjunto controlado de árbol, costos unitarios y extremos en esquinas, las medianas observadas de expansiones para BFS fueron 182 (15×15), 492 (25×25) y 1267.5 (40×40); para IDDFS fueron 5493, 24394 y 102420. BFS y UCS devolvieron la misma profundidad en el árbol y Lee coincidió con ambos, como predice la ruta única y el costo unitario. IDDFS expande mucho más por sus reinicios, aunque conserva una frontera menor que BFS. El tiempo no debe compararse solo: varía con el equipo, la carga y la instrumentación; se reportan mediana e IQR de 30 semillas.

En grafos ponderados, la comparación de costo óptimo corresponde a UCS. BFS y Lee encuentran menos pasos, no necesariamente menor costo; la búsqueda bidireccional implementada aquí también es BFS y solo se incluye en condiciones unitarias.

## 12. Preguntas para la defensa

El docente seleccionará algunas al azar:

1. Muestre en memoria la diferencia entre frontera y alcanzados.
2. Cambie BFS a DFS modificando solamente la política apropiada.
3. Explique un caso en el que marcar visitados demasiado tarde duplique trabajo.
4. Explique por qué terminar UCS al generar la meta puede ser incorrecto.
5. Reconstruya un camino usando padres sin almacenar caminos completos.
6. Muestre por qué dos BFS bidireccionales requieren orientar correctamente los padres.
7. Explique por qué Lee no es un rayo geométrico.
8. Adapte el problema a una meta múltiple.
9. Introduzca una celda bloqueada durante la ejecución y analice qué debe recalcularse.
10. Compare `SearchProblem` de SimpleAI con `Problem` de AIMA-Python.
11. Diagnostique una discrepancia de métricas entre dos versiones.
12. Argumente por qué un laberinto perfecto oculta diferencias de calidad de ruta.

## 13. Entrega

El cuaderno final debe contener:

1. auditoría del generador;
2. formulación formal;
3. versión desde cero;
4. versión SimpleAI;
5. versión AIMA-Python;
6. pruebas automáticas;
7. extensión con ciclos y costos;
8. protocolo experimental y gráficas;
9. análisis teórico;
10. conclusiones y referencias;
11. enlace al historial de trabajo o bitácora incorporada.

Todo el cuaderno debe ejecutarse desde el inicio en un entorno limpio. Las celdas fuera de orden, dependencias implícitas y resultados pegados manualmente se consideran defectos reproducibles.

### Rúbrica

| Criterio | Peso |
|---|---:|
| Modelado, invariantes y auditoría de Hunt-and-Kill | 10% |
| Versión desde cero y estructuras de datos | 25% |
| Adaptación correcta a SimpleAI | 12% |
| Adaptación correcta a AIMA-Python | 13% |
| Pruebas, casos límite y concordancia | 15% |
| Experimentos, métricas y visualización | 12% |
| Análisis teórico y conclusiones | 8% |
| Defensa, trazabilidad y calidad del cuaderno | 5% |

Una solución que produzca un camino pero no satisfaga el contrato, las pruebas o la defensa no se considera completa.

## 14. Referencias de partida

- Russell, S. J. y Norvig, P. *Artificial Intelligence: A Modern Approach*, 4.ª edición, capítulo 3.
- SimpleAI, documentación de problemas y búsqueda tradicional: <https://github.com/simpleai-team/simpleai/blob/master/docs/search_problems.rst>
- SimpleAI, implementación oficial de búsqueda tradicional: <https://github.com/simpleai-team/simpleai/blob/master/simpleai/search/traditional.py>
- AIMA-Python, repositorio oficial: <https://github.com/aimacode/aima-python>
- AIMA-Python, módulo de búsqueda: <https://github.com/aimacode/aima-python/blob/master/aima/search.py>
- C. Y. Lee, “An Algorithm for Path Connections and Its Applications”, 1961.
- A. Adamatzky, “Physical maze solvers. All twelve prototypes implement 1961 Lee algorithm”, 2016: <https://arxiv.org/abs/1601.04672>

Las referencias orientan el estudio de las interfaces; no sustituyen la explicación de las decisiones tomadas.